In [ ]:
# ============================================================
# NOTEBOOK 03 — DATA CLEANING & WRANGLING
# 03.1 — IMPORT LIBRARIES & PROJECT CONFIGURATION
# ============================================================

import os
import sys
import pandas as pd
import numpy as np

from pathlib import Path

# ------------------------------------------------------------
# LOAD LOCAL CONFIGURATION
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"G:\.shortcut-targets-by-id\1Xk-_mYwUHPWs3pKBzHBs1yQGslb7jsqJ\PROJECT SAINS DATA"
)

sys.path.insert(0, str(PROJECT_DIR))

from config_local import (
    DATA_DIR,
    COMPOUND_DIR,
    MAOB_DIR,
    ALPHA_SYNUCLEIN_DIR,
    OUTPUT_DIR,
    CLEANED_DATA_DIR
)

# ------------------------------------------------------------
# VALIDATE INPUT DIRECTORIES
# ------------------------------------------------------------

directories = {
    "PROJECT": PROJECT_DIR,
    "DATA": DATA_DIR,
    "COMPOUND": COMPOUND_DIR,
    "MAO-B": MAOB_DIR,
    "Alpha-Synuclein": ALPHA_SYNUCLEIN_DIR,
    "OUTPUT": OUTPUT_DIR,
    "CLEANED_DATA": CLEANED_DATA_DIR
}

print("=" * 65)
print("PROJECT CONFIGURATION VALIDATION")
print("=" * 65)

for name, path in directories.items():
    print(
        f"{name:<20}:",
        "OK" if path.is_dir() else "NOT FOUND"
    )

print("\nPROJECT DIR:", PROJECT_DIR)
print("OUTPUT DIR :", OUTPUT_DIR)

In [ ]:
# ============================================================
# NOTEBOOK 03 — DATA CLEANING & WRANGLING
# 03.2 — LOAD RAW DATASETS
# ============================================================

# ------------------------------------------------------------
# DEFINE INPUT FILES
# ------------------------------------------------------------

dataset_files = {
    "pubchem_compounds": COMPOUND_DIR / "pubchem_compounds.csv",
    "validated_compounds": COMPOUND_DIR / "validated_compounds.csv",
    "compound_identity_validation": COMPOUND_DIR / "compound_identity_validation.csv",
    "compound_duplicate_validation": COMPOUND_DIR / "compound_duplicate_validation.csv",

    "maob_target_metadata": MAOB_DIR / "maob_target_metadata.csv",
    "maob_activity": MAOB_DIR / "maob_activity_raw.csv",

    "alpha_target_metadata": ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_target_metadata.csv",
    "alpha_activity": ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_activity_raw.csv",
    "alpha_assay_metadata": ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_assay_metadata_raw.csv",
}

# ------------------------------------------------------------
# LOAD DATASETS
# ------------------------------------------------------------

datasets = {}
missing_files = []

for name, file_path in dataset_files.items():

    if file_path.is_file():
        datasets[name] = pd.read_csv(file_path)
        print(f"[FOUND] {name:<35} {datasets[name].shape}")

    else:
        missing_files.append(name)
        print(f"[MISSING] {name:<33} {file_path}")

# ------------------------------------------------------------
# DATASET INVENTORY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET LOADING SUMMARY")
print("=" * 70)

print("Successfully loaded :", len(datasets))
print("Missing files       :", len(missing_files))

for name, df in datasets.items():
    print(f"{name:<35} {len(df):>8,} rows | {len(df.columns):>3} columns")

In [ ]:
# ============================================================
# 03.3 — CHECK ALPHA-SYNUCLEIN DIRECTORY
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN DIRECTORY INSPECTION")
print("=" * 65)

print("Directory:", ALPHA_SYNUCLEIN_DIR)
print("Exists   :", ALPHA_SYNUCLEIN_DIR.exists())

print("\nFiles available:")

for file in ALPHA_SYNUCLEIN_DIR.iterdir():
    print(f"{file.name:<50} {file.stat().st_size:,} bytes")

In [ ]:
# ============================================================
# SEARCH MISSING ASSAY METADATA
# ============================================================

target_filename = "alpha_synuclein_assay_metadata_raw.csv"

print("Searching:", PROJECT_DIR)

found_files = list(PROJECT_DIR.rglob(target_filename))

if found_files:
    print("\nFILE FOUND:")
    for file in found_files:
        print(file)
else:
    print("\nFILE NOT FOUND IN PROJECT DIRECTORY")

In [ ]:
# ============================================================
# 03.4 — DATASET STRUCTURE INSPECTION
# ============================================================

for name, df in datasets.items():

    print("\n" + "=" * 70)
    print(f"DATASET: {name.upper()}")
    print("=" * 70)

    print("Shape:", df.shape)

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes.value_counts())

    print("\nFirst 3 rows:")
    display(df.head(3))

In [ ]:
# ============================================================
# 03.5 — MISSING VALUES AUDIT
# ============================================================

activity_datasets = {
    "MAO-B": datasets["maob_activity"],
    "Alpha-Synuclein": datasets["alpha_activity"]
}

missing_summary = []

for target_name, df in activity_datasets.items():

    print("\n" + "=" * 70)
    print(f"MISSING VALUES AUDIT — {target_name}")
    print("=" * 70)

    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    summary = pd.DataFrame({
        "Missing_Count": missing_count,
        "Missing_Percentage": missing_percent
    })

    summary = summary.sort_values(
        "Missing_Percentage",
        ascending=False
    )

    display(summary[summary["Missing_Count"] > 0])

    missing_summary.append(
        summary.assign(Target=target_name)
    )

# Gabungkan ringkasan kedua target
missing_summary_df = pd.concat(missing_summary)

print("\nMissing values audit completed.")

In [ ]:
# ============================================================
# CELL 03.6 — ENDPOINT AND UNIT DISTRIBUTION AUDIT
# ============================================================

audit_columns = [
    "type",
    "standard_type",
    "relation",
    "standard_relation",
    "units",
    "standard_units",
    "assay_type",
    "bao_endpoint"
]

activity_datasets = {
    "MAO-B": datasets["maob_activity"],
    "Alpha-Synuclein": datasets["alpha_activity"]
}

for target_name, df in activity_datasets.items():

    print("\n" + "=" * 75)
    print(f"ENDPOINT AND UNIT AUDIT — {target_name}")
    print("=" * 75)

    for col in audit_columns:

        if col not in df.columns:
            continue

        print(f"\n--- {col} ---")

        distribution = (
            df[col]
            .fillna("[MISSING]")
            .value_counts(dropna=False)
            .head(20)
            .rename_axis(col)
            .reset_index(name="Count")
        )

        distribution["Percentage"] = (
            distribution["Count"] / len(df) * 100
        ).round(2)

        display(distribution)

print("\nEndpoint and unit audit completed.")

In [ ]:

# ============================================================
# CELL 03.7 — DETAIL AUDIT ENDPOINT UTAMA
# ============================================================

endpoint_groups = {
    "MAO-B IC50": {
        "df": datasets["maob_activity"],
        "endpoint": "IC50"
    },
    "Alpha-Synuclein Potency": {
        "df": datasets["alpha_activity"],
        "endpoint": "Potency"
    },
    "Alpha-Synuclein IC50": {
        "df": datasets["alpha_activity"],
        "endpoint": "IC50"
    }
}

for group_name, config in endpoint_groups.items():

    df = config["df"]
    endpoint = config["endpoint"]

    subset = df[
        df["standard_type"].astype(str).str.strip() == endpoint
    ].copy()

    print("\n" + "=" * 75)
    print(f"DETAIL AUDIT — {group_name}")
    print("=" * 75)

    print(f"Total records       : {len(subset):,}")
    print(f"Unique molecules    : {subset['molecule_chembl_id'].nunique():,}")
    print(f"Unique assays       : {subset['assay_chembl_id'].nunique():,}")
    print(f"Unique documents    : {subset['document_chembl_id'].nunique():,}")

    # Distribusi satuan, relasi, dan assay
    for column in [
        "standard_units",
        "standard_relation",
        "assay_type",
        "bao_endpoint"
    ]:
        print(f"\n--- {column} ---")
        print(
            subset[column]
            .fillna("[MISSING]")
            .value_counts()
            .head(15)
            .to_string()
        )

    # Pemeriksaan nilai numerik
    numeric_values = pd.to_numeric(
        subset["standard_value"],
        errors="coerce"
    )

    print("\n--- STANDARD VALUE ---")
    print("Valid numeric       :", numeric_values.notna().sum())
    print("Missing/non-numeric :", numeric_values.isna().sum())

    if numeric_values.notna().any():
        print(numeric_values.describe())

    # Pemeriksaan deskripsi assay
    print("\n--- ASSAY DESCRIPTION (TOP 15) ---")
    print(
        subset["assay_description"]
        .fillna("[MISSING]")
        .value_counts()
        .head(15)
        .to_string()
    )

print("\nDetail endpoint audit completed.")


In [ ]:
# ============================================================
# CELL 03.7A — RINGKASAN DETAIL ENDPOINT
# ============================================================

endpoint_groups = {
    "MAO-B IC50": (datasets["maob_activity"], "IC50"),
    "Alpha-Synuclein Potency": (datasets["alpha_activity"], "Potency"),
    "Alpha-Synuclein IC50": (datasets["alpha_activity"], "IC50")
}

for group_name, (df, endpoint) in endpoint_groups.items():

    subset = df[
        df["standard_type"].astype(str).str.strip() == endpoint
    ].copy()

    print("\n" + "=" * 70)
    print(f"{group_name}")
    print("=" * 70)

    print("Total records:", len(subset))
    print("Unique assays:", subset["assay_chembl_id"].nunique())
    print("Unique documents:", subset["document_chembl_id"].nunique())

    print("\n[UNIT × RELATION]")
    print(pd.crosstab(
        subset["standard_units"].fillna("MISSING"),
        subset["standard_relation"].fillna("MISSING")
    ).to_string())

    values = pd.to_numeric(
        subset["standard_value"], errors="coerce"
    )

    print("\n[NUMERIC VALUE]")
    print("Valid:", values.notna().sum())
    print("Missing/non-numeric:", values.isna().sum())
    print("Zero or negative:", (values <= 0).sum())

    if values.notna().any():
        print(values.describe(
            percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
        ).to_string())

    print("\n[ASSAY DESCRIPTION — TOP 10]")

    descriptions = (
        subset["assay_description"]
        .fillna("MISSING")
        .value_counts()
        .head(10)
    )

    for description, count in descriptions.items():
        print(f"- ({count}) {str(description)[:180]}")

    print("\n[TYPE VS STANDARD_TYPE]")
    print("Original type:")
    print(subset["type"].value_counts(dropna=False).head(10).to_string())

    print("\nStandard type:")
    print(subset["standard_type"].value_counts(dropna=False).head(10).to_string())

In [ ]:
# ============================================================
# CELL 03.8 — ASSAY CONSISTENCY AUDIT
# ============================================================

audit_groups = {
    "MAO-B IC50 (nM, exact)": {
        "df": datasets["maob_activity"],
        "endpoint": "IC50",
        "unit": "nM",
        "relation": "="
    },
    "Alpha-Synuclein IC50 (nM, exact)": {
        "df": datasets["alpha_activity"],
        "endpoint": "IC50",
        "unit": "nM",
        "relation": "="
    }
}

for group_name, config in audit_groups.items():

    df = config["df"]

    subset = df[
        (df["standard_type"] == config["endpoint"]) &
        (df["standard_units"] == config["unit"]) &
        (df["standard_relation"] == config["relation"])
    ].copy()

    print("\n" + "=" * 80)
    print(group_name)
    print("=" * 80)

    print("Records:", len(subset))
    print("Unique assays:", subset["assay_chembl_id"].nunique())
    print("Unique documents:", subset["document_chembl_id"].nunique())

    # Ringkasan kontribusi setiap assay
    assay_summary = (
        subset.groupby("assay_chembl_id", dropna=False)
        .agg(
            record_count=("molecule_chembl_id", "size"),
            unique_molecules=("molecule_chembl_id", "nunique"),
            document_id=("document_chembl_id", "first"),
            assay_type=("assay_type", "first"),
            assay_description=("assay_description", "first")
        )
        .sort_values("record_count", ascending=False)
    )

    print("\n--- TOP 20 ASSAYS ---")

    for assay_id, row in assay_summary.head(20).iterrows():
        print(f"\nAssay ID       : {assay_id}")
        print(f"Records        : {row['record_count']}")
        print(f"Unique molecules: {row['unique_molecules']}")
        print(f"Document ID    : {row['document_id']}")
        print(f"Assay type     : {row['assay_type']}")
        print(
            "Description    :",
            str(row["assay_description"])[:300]
        )

    print("\n--- ASSAY RECORD DISTRIBUTION ---")
    print(assay_summary["record_count"].describe().to_string())

    print("\n--- ORIGINAL TYPE DISTRIBUTION ---")
    print(subset["type"].value_counts(dropna=False).to_string())

print("\nAssay consistency audit completed.")

In [ ]:
# ============================================================
# CELL 03.10 — ORIGINAL VS STANDARDIZED ACTIVITY AUDIT
# ============================================================

maob = datasets["maob_activity"].copy()

subset = maob[
    (maob["standard_type"] == "IC50") &
    (maob["standard_units"] == "nM") &
    (maob["standard_relation"] == "=")
].copy()

print("=" * 75)
print("ORIGINAL VS STANDARDIZED ACTIVITY — MAO-B")
print("=" * 75)

for original_type, group in subset.groupby("type", dropna=False):

    print(f"\n{'-' * 60}")
    print("Original type:", original_type)
    print("Records:", len(group))

    print("\nOriginal units:")
    print(group["units"].value_counts(dropna=False).to_string())

    print("\nOriginal relation:")
    print(group["relation"].value_counts(dropna=False).to_string())

    print("\nOriginal value:")
    print(
        pd.to_numeric(group["value"], errors="coerce")
        .describe()
        .to_string()
    )

    print("\nStandardized value:")
    print(
        pd.to_numeric(group["standard_value"], errors="coerce")
        .describe()
        .to_string()
    )

print("\n--- SAMPLE RECORDS ---")

columns_to_show = [
    "molecule_chembl_id",
    "type",
    "relation",
    "value",
    "units",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "assay_chembl_id"
]

display(subset[columns_to_show].groupby("type", dropna=False).head(5))

print("\nOriginal vs standardized audit completed.")

In [ ]:
# ============================================================
# CELL 03.11 — STANDARDIZATION VERIFICATION
# ============================================================

maob = datasets["maob_activity"]

subset = maob[
    (maob["standard_type"] == "IC50") &
    (maob["standard_units"] == "nM") &
    (maob["standard_relation"] == "=")
].copy()

columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "type",
    "value",
    "units",
    "standard_type",
    "standard_value",
    "standard_units"
]

for original_type in ["IC50", "Log IC50", "pIC50"]:

    group = subset[subset["type"] == original_type]

    print("\n" + "=" * 70)
    print(f"STANDARDIZATION CHECK — {original_type}")
    print("=" * 70)

    sample = group[columns].head(10).copy()

    display(sample)

    original = pd.to_numeric(group["value"], errors="coerce")
    standardized = pd.to_numeric(
        group["standard_value"], errors="coerce"
    )

    if original_type == "pIC50":
        expected = 10 ** (9 - original)

    elif original_type == "Log IC50":
        expected = 10 ** (original + 9)

    else:
        expected = None

    if expected is not None:
        comparison = pd.DataFrame({
            "original_value": original,
            "expected_nM": expected,
            "standard_value": standardized
        })

        comparison["relative_error"] = (
            (comparison["standard_value"] - comparison["expected_nM"]).abs()
            / comparison["expected_nM"]
        )

        print("\nConversion comparison:")
        display(comparison.head(10))

        print(
            "Median relative error:",
            comparison["relative_error"].median()
        )

print("\nStandardization verification completed.")

In [ ]:

# ============================================================
# CELL 03.11 — REPEATED MOLECULE & ACTIVITY AUDIT
# ============================================================

maob = datasets["maob_activity"].copy()

exact_ic50 = maob[
    (maob["standard_type"] == "IC50") &
    (maob["standard_units"] == "nM") &
    (maob["standard_relation"] == "=")
].copy()

exact_ic50["standard_value"] = pd.to_numeric(
    exact_ic50["standard_value"],
    errors="coerce"
)

exact_ic50 = exact_ic50[
    exact_ic50["standard_value"] > 0
].copy()

print("=" * 70)
print("REPEATED MOLECULE AUDIT — MAO-B")
print("=" * 70)

# 1. Pengulangan molekul
molecule_counts = (
    exact_ic50.groupby("molecule_chembl_id")
    .size()
    .sort_values(ascending=False)
)

print("\nTotal records:", len(exact_ic50))
print("Unique molecules:", exact_ic50["molecule_chembl_id"].nunique())
print("Molecules tested more than once:",
      (molecule_counts > 1).sum())

print("\nTop 15 repeated molecules:")
print(molecule_counts.head(15).to_string())

# 2. Pengukuran berulang dalam assay yang sama
within_assay = (
    exact_ic50.groupby(
        ["molecule_chembl_id", "assay_chembl_id"]
    )
    .size()
)

print("\nRepeated molecule within same assay:")
print("Groups with repeated measurements:",
      (within_assay > 1).sum())

# 3. Pengukuran dalam assay berbeda
assay_per_molecule = (
    exact_ic50.groupby("molecule_chembl_id")["assay_chembl_id"]
    .nunique()
)

print("\nMolecules tested in multiple assays:")
print((assay_per_molecule > 1).sum())

# 4. Distribusi jumlah pengukuran
print("\nRecords per molecule:")
print(molecule_counts.describe().to_string())

# 5. Duplikasi record
duplicate_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

print("\nExact duplicate records:",
      exact_ic50.duplicated(subset=duplicate_columns).sum())

print("\nRepeated molecule audit completed.")



In [ ]:
# ============================================================
# CELL 03.12 — INSPECT EXACT DUPLICATES
# ============================================================

df_maob_raw = datasets["maob_activity"].copy()

# Cari duplikasi berdasarkan seluruh kolom
duplicate_mask = df_maob_raw.duplicated(keep=False)

df_exact_duplicates = (
    df_maob_raw.loc[duplicate_mask]
    .sort_values(
        by=[
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value"
        ]
    )
)

print("=" * 70)
print("EXACT DUPLICATE INSPECTION — MAO-B")
print("=" * 70)

print("Total duplicate rows:", len(df_exact_duplicates))

display(df_exact_duplicates)

# Simpan kandidat duplikasi untuk dokumentasi audit
df_exact_duplicates.to_csv(
    CLEANED_DATA_DIR / "maob_exact_duplicate_candidates.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 03.13 — DUPLICATE KEY RECHECK
# ============================================================

df_maob = datasets["maob_activity"].copy()

# Filter subset aktivitas yang sedang kita audit
df_check = df_maob[
    (df_maob["standard_type"] == "IC50") &
    (df_maob["standard_units"] == "nM") &
    (df_maob["standard_relation"] == "=")
].copy()

# Definisi duplikasi berdasarkan identitas pengukuran
duplicate_keys = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

# Identifikasi semua baris dengan kombinasi key berulang
duplicate_mask = df_check.duplicated(
    subset=duplicate_keys,
    keep=False
)

df_duplicate_candidates = (
    df_check.loc[duplicate_mask]
    .sort_values(duplicate_keys)
)

print("=" * 70)
print("DUPLICATE KEY RECHECK — MAO-B")
print("=" * 70)

print("Total candidate rows:", len(df_duplicate_candidates))

display(
    df_duplicate_candidates[
        duplicate_keys + [
            "activity_id",
            "record_id",
            "document_chembl_id",
            "assay_description"
        ]
    ]
)

# Preparation Cleaning

In [ ]:

# ============================================================
# CELL 03.14 — CLEANING JUSTIFICATION & DECISION
# PROJECT: AURICULARIA POLYTRICHA × PARKINSON
# TARGET: HUMAN MAO-B
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. LOAD RAW DATA
# ------------------------------------------------------------

df_audit = datasets["maob_activity"].copy()

df_audit["standard_value"] = pd.to_numeric(
    df_audit["standard_value"],
    errors="coerce"
)

total_records = len(df_audit)

print("=" * 75)
print("CLEANING JUSTIFICATION — MAO-B")
print("=" * 75)

print(f"Total original records : {total_records:,}")
print(f"Total columns          : {len(df_audit.columns)}")


# ------------------------------------------------------------
# 2. MISSING VALUE AUDIT
# ------------------------------------------------------------

missing_summary = pd.DataFrame({
    "Missing_Count": df_audit.isna().sum(),
    "Missing_Percentage": (
        df_audit.isna().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

print("\n" + "=" * 75)
print("A. MISSING VALUE EVIDENCE")
print("=" * 75)

display(missing_summary.head(20))


# ------------------------------------------------------------
# 3. ENDPOINT AUDIT
# ------------------------------------------------------------

endpoint_summary = (
    df_audit["standard_type"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("Standard_Type")
    .reset_index(name="Record_Count")
)

endpoint_summary["Percentage"] = (
    endpoint_summary["Record_Count"] / total_records * 100
).round(2)

print("\n" + "=" * 75)
print("B. ENDPOINT HETEROGENEITY")
print("=" * 75)

display(endpoint_summary)


# ------------------------------------------------------------
# 4. UNIT AUDIT
# ------------------------------------------------------------

unit_summary = (
    df_audit["standard_units"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("Standard_Units")
    .reset_index(name="Record_Count")
)

unit_summary["Percentage"] = (
    unit_summary["Record_Count"] / total_records * 100
).round(2)

print("\n" + "=" * 75)
print("C. ACTIVITY UNIT HETEROGENEITY")
print("=" * 75)

display(unit_summary)


# ------------------------------------------------------------
# 5. RELATION AUDIT
# ------------------------------------------------------------

relation_summary = (
    df_audit["standard_relation"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("Standard_Relation")
    .reset_index(name="Record_Count")
)

relation_summary["Percentage"] = (
    relation_summary["Record_Count"] / total_records * 100
).round(2)

print("\n" + "=" * 75)
print("D. ACTIVITY RELATION AUDIT")
print("=" * 75)

display(relation_summary)


# ------------------------------------------------------------
# 6. NUMERIC VALUE AUDIT
# ------------------------------------------------------------

numeric_values = df_audit["standard_value"]

numeric_audit = pd.DataFrame({
    "Condition": [
        "Missing / non-numeric values",
        "Zero values",
        "Negative values",
        "Positive numeric values"
    ],
    "Record_Count": [
        numeric_values.isna().sum(),
        (numeric_values == 0).sum(),
        (numeric_values < 0).sum(),
        (numeric_values > 0).sum()
    ]
})

numeric_audit["Percentage"] = (
    numeric_audit["Record_Count"] / total_records * 100
).round(2)

print("\n" + "=" * 75)
print("E. NUMERIC VALUE VALIDITY")
print("=" * 75)

display(numeric_audit)


# ------------------------------------------------------------
# 7. REPEATED MOLECULE AUDIT
# ------------------------------------------------------------

molecule_counts = (
    df_audit["molecule_chembl_id"]
    .value_counts()
)

repeated_molecules = molecule_counts[
    molecule_counts > 1
]

print("\n" + "=" * 75)
print("F. REPEATED MOLECULE EVIDENCE")
print("=" * 75)

print("Unique molecules:", df_audit["molecule_chembl_id"].nunique())
print("Repeated molecules:", len(repeated_molecules))
print("Records belonging to repeated molecules:",
      int(molecule_counts[molecule_counts > 1].sum()))

display(repeated_molecules.head(15).to_frame(
    name="Record_Count"
))


# ------------------------------------------------------------
# 8. DUPLICATE CANDIDATE AUDIT
# ------------------------------------------------------------

duplicate_keys = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

# Fokus pada subset IC50 exact dalam nM
df_ic50_candidate = df_audit[
    (df_audit["standard_type"] == "IC50") &
    (df_audit["standard_units"] == "nM") &
    (df_audit["standard_relation"] == "=") &
    (df_audit["standard_value"] > 0)
].copy()

duplicate_mask = df_ic50_candidate.duplicated(
    subset=duplicate_keys,
    keep=False
)

df_duplicate_evidence = df_ic50_candidate[
    duplicate_mask
].copy()

print("\n" + "=" * 75)
print("G. DUPLICATE CANDIDATE EVIDENCE")
print("=" * 75)

print("IC50 exact subset:", len(df_ic50_candidate))
print("Candidate duplicate rows:", len(df_duplicate_evidence))

if len(df_duplicate_evidence) > 0:
    display(
        df_duplicate_evidence[
            duplicate_keys + [
                "activity_id",
                "record_id",
                "document_chembl_id"
            ]
        ].sort_values(duplicate_keys)
    )


# ------------------------------------------------------------
# 9. CLEANING DECISION TABLE
# ------------------------------------------------------------

decision_table = pd.DataFrame([
    {
        "Issue": "Missing values",
        "Evidence": (
            f"{int(df_audit.isna().any(axis=1).sum()):,} records "
            "memiliki minimal satu missing value"
        ),
        "Cleaning_Decision": (
            "Evaluasi berdasarkan kebutuhan kolom; "
            "jangan menghapus seluruh baris secara otomatis"
        )
    },
    {
        "Issue": "Endpoint heterogeneity",
        "Evidence": (
            f"{df_audit['standard_type'].nunique(dropna=True)} "
            "jenis endpoint ditemukan"
        ),
        "Cleaning_Decision": (
            "Pilih IC50 untuk dataset aktivitas utama MAO-B"
        )
    },
    {
        "Issue": "Unit heterogeneity",
        "Evidence": (
            f"{df_audit['standard_units'].nunique(dropna=True)} "
            "jenis satuan ditemukan"
        ),
        "Cleaning_Decision": (
            "Gunakan nM untuk dataset aktivitas utama"
        )
    },
    {
        "Issue": "Censored relations",
        "Evidence": (
            f"{int((df_audit['standard_relation'] != '=').sum()):,} "
            "record tidak memiliki relation exact"
        ),
        "Cleaning_Decision": (
            "Pisahkan relation selain '='; jangan perlakukan "
            "nilai tersensor sebagai nilai eksak"
        )
    },
    {
        "Issue": "Invalid numeric values",
        "Evidence": (
            f"{int((numeric_values.isna() | (numeric_values <= 0)).sum()):,} "
            "record missing, non-numeric, atau tidak positif"
        ),
        "Cleaning_Decision": (
            "Keluarkan dari dataset numerik utama dan dokumentasikan"
        )
    },
    {
        "Issue": "Repeated molecules",
        "Evidence": (
            f"{len(repeated_molecules):,} molekul memiliki "
            "lebih dari satu record"
        ),
        "Cleaning_Decision": (
            "Pertahankan; evaluasi assay dan pengukuran berulang"
        )
    },
    {
        "Issue": "Duplicate candidates",
        "Evidence": (
            f"{len(df_duplicate_evidence):,} baris kandidat "
            "berdasarkan kombinasi key"
        ),
        "Cleaning_Decision": (
            "Pertahankan sementara; periksa identitas record "
            "sebelum memutuskan penghapusan"
        )
    }
])

print("\n" + "=" * 75)
print("H. CLEANING DECISION SUMMARY")
print("=" * 75)

display(decision_table)


# ------------------------------------------------------------
# 10. SAVE AUDIT EVIDENCE
# ------------------------------------------------------------

CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_missing_audit.csv"
)

endpoint_summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_endpoint_audit.csv",
    index=False
)

unit_summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_unit_audit.csv",
    index=False
)

relation_summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_relation_audit.csv",
    index=False
)

numeric_audit.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_numeric_audit.csv",
    index=False
)

decision_table.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_decision.csv",
    index=False
)

df_duplicate_evidence.to_csv(
    CLEANED_DATA_DIR / "maob_duplicate_candidates.csv",
    index=False
)

print("\n" + "=" * 75)
print("CLEANING JUSTIFICATION COMPLETED")
print("=" * 75)
print("Audit evidence saved to:", CLEANED_DATA_DIR)


In [ ]:
# ============================================================
# CELL 03.14 — PREPARE MAO-B CLEANING
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Ambil data mentah
df_maob_cleaning = datasets["maob_activity"].copy()

# Pastikan kolom numerik dapat dibaca
df_maob_cleaning["standard_value"] = pd.to_numeric(
    df_maob_cleaning["standard_value"],
    errors="coerce"
)

print("=" * 70)
print("MAO-B CLEANING PREPARATION")
print("=" * 70)

print("Original records:", len(df_maob_cleaning))
print("Original columns:", len(df_maob_cleaning.columns))

# Pastikan output directory tersedia
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", CLEANED_DATA_DIR)
print("Directory exists:", CLEANED_DATA_DIR.exists())

# Validasi kolom yang dibutuhkan
required_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

missing_columns = [
    col for col in required_columns
    if col not in df_maob_cleaning.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

print("\nPreparation completed successfully.")

In [ ]:

# ============================================================
# CELL 03.15 — MAO-B CLEANING IMPLEMENTATION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. LOAD RAW DATA
# ------------------------------------------------------------

df_raw = datasets["maob_activity"].copy()

df_raw["standard_value"] = pd.to_numeric(
    df_raw["standard_value"],
    errors="coerce"
)

total_before = len(df_raw)

print("=" * 75)
print("MAO-B CLEANING IMPLEMENTATION")
print("=" * 75)

print("Original records:", total_before)


# ------------------------------------------------------------
# 2. DEFINE CLEANING CRITERIA
# ------------------------------------------------------------

condition_endpoint = (
    df_raw["standard_type"].eq("IC50")
)

condition_unit = (
    df_raw["standard_units"].eq("nM")
)

condition_relation = (
    df_raw["standard_relation"].eq("=")
)

condition_numeric = (
    df_raw["standard_value"].notna()
)

condition_positive = (
    df_raw["standard_value"] > 0
)

condition_molecule = (
    df_raw["molecule_chembl_id"].notna()
)

condition_assay = (
    df_raw["assay_chembl_id"].notna()
)

# Gabungan seluruh kriteria
valid_mask = (
    condition_endpoint &
    condition_unit &
    condition_relation &
    condition_numeric &
    condition_positive &
    condition_molecule &
    condition_assay
)


# ------------------------------------------------------------
# 3. CREATE MAIN CLEANED DATASET
# ------------------------------------------------------------

df_maob_cleaned = df_raw.loc[valid_mask].copy()

# Tandai kandidat duplikasi
duplicate_keys = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

df_maob_cleaned["duplicate_candidate"] = (
    df_maob_cleaned.duplicated(
        subset=duplicate_keys,
        keep=False
    )
)

print("\nMain cleaned dataset:", len(df_maob_cleaned))


# ------------------------------------------------------------
# 4. CREATE EXCLUSION REASONS
# ------------------------------------------------------------

df_excluded = df_raw.loc[~valid_mask].copy()

def get_exclusion_reasons(row):

    reasons = []

    if pd.isna(row["standard_type"]):
        reasons.append("Missing endpoint")
    elif row["standard_type"] != "IC50":
        reasons.append("Endpoint is not IC50")

    if pd.isna(row["standard_units"]):
        reasons.append("Missing standard unit")
    elif row["standard_units"] != "nM":
        reasons.append("Unit is not nM")

    if pd.isna(row["standard_relation"]):
        reasons.append("Missing relation")
    elif row["standard_relation"] != "=":
        reasons.append("Censored/non-exact relation")

    if pd.isna(row["standard_value"]):
        reasons.append("Missing/non-numeric activity value")
    elif row["standard_value"] <= 0:
        reasons.append("Non-positive IC50 value")

    if pd.isna(row["molecule_chembl_id"]):
        reasons.append("Missing molecule ID")

    if pd.isna(row["assay_chembl_id"]):
        reasons.append("Missing assay ID")

    return "; ".join(reasons)


df_excluded["exclusion_reason"] = df_excluded.apply(
    get_exclusion_reasons,
    axis=1
)


# ------------------------------------------------------------
# 5. VALIDATION SUMMARY
# ------------------------------------------------------------

total_after = len(df_maob_cleaned)
total_excluded = len(df_excluded)

summary = pd.DataFrame({
    "Category": [
        "Original records",
        "Cleaned records",
        "Excluded records"
    ],
    "Record_Count": [
        total_before,
        total_after,
        total_excluded
    ]
})

summary["Percentage"] = (
    summary["Record_Count"] / total_before * 100
).round(2)

print("\n" + "=" * 75)
print("CLEANING SUMMARY")
print("=" * 75)

display(summary)

# Pastikan semua record terpartisi
assert total_after + total_excluded == total_before

print("Partition validation: PASSED")


# ------------------------------------------------------------
# 6. CLEANED DATA VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CLEANED DATA QUALITY CHECK")
print("=" * 75)

print("Endpoint unique:",
      df_maob_cleaned["standard_type"].unique())

print("Unit unique:",
      df_maob_cleaned["standard_units"].unique())

print("Relation unique:",
      df_maob_cleaned["standard_relation"].unique())

print("Missing activity values:",
      df_maob_cleaned["standard_value"].isna().sum())

print("Non-positive activity values:",
      (df_maob_cleaned["standard_value"] <= 0).sum())

print("Unique molecules:",
      df_maob_cleaned["molecule_chembl_id"].nunique())

print("Unique assays:",
      df_maob_cleaned["assay_chembl_id"].nunique())

print("Duplicate candidate rows:",
      df_maob_cleaned["duplicate_candidate"].sum())


# ------------------------------------------------------------
# 7. EXCLUSION REASON SUMMARY
# ------------------------------------------------------------

exclusion_summary = (
    df_excluded["exclusion_reason"]
    .value_counts()
    .rename_axis("Exclusion_Reason")
    .reset_index(name="Record_Count")
)

print("\n" + "=" * 75)
print("EXCLUSION REASON SUMMARY")
print("=" * 75)

display(exclusion_summary)


# ------------------------------------------------------------
# 8. SAVE CLEANING OUTPUTS
# ------------------------------------------------------------

CLEANED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df_maob_cleaned.to_csv(
    CLEANED_DATA_DIR / "maob_activity_cleaned.csv",
    index=False
)

df_excluded.to_csv(
    CLEANED_DATA_DIR / "maob_activity_excluded.csv",
    index=False
)

summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_summary.csv",
    index=False
)

exclusion_summary.to_csv(
    CLEANED_DATA_DIR / "maob_exclusion_summary.csv",
    index=False
)

print("\n" + "=" * 75)
print("MAO-B CLEANING OUTPUT SAVED")
print("=" * 75)

print("Cleaned dataset:",
      CLEANED_DATA_DIR / "maob_activity_cleaned.csv")

print("Excluded dataset:",
      CLEANED_DATA_DIR / "maob_activity_excluded.csv")


In [ ]:
import pandas as pd
import numpy as np

# Load saved cleaned dataset
cleaned_path = CLEANED_DATA_DIR / "maob_activity_cleaned.csv"
excluded_path = CLEANED_DATA_DIR / "maob_activity_excluded.csv"

df_maob_cleaned = pd.read_csv(cleaned_path)
df_maob_excluded = pd.read_csv(excluded_path)

print("=" * 70)
print("FINAL VALIDATION — MAO-B CLEANED DATA")
print("=" * 70)

# 1. Dataset dimensions
print("\nA. DATASET DIMENSIONS")
print("Cleaned:", df_maob_cleaned.shape)
print("Excluded:", df_maob_excluded.shape)

# 2. Required column validation
required_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

missing_columns = [
    col for col in required_columns
    if col not in df_maob_cleaned.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

# 3. Activity value validation
df_maob_cleaned["standard_value"] = pd.to_numeric(
    df_maob_cleaned["standard_value"],
    errors="coerce"
)

assert df_maob_cleaned["standard_value"].notna().all()
assert (df_maob_cleaned["standard_value"] > 0).all()

# 4. Endpoint consistency
assert df_maob_cleaned["standard_type"].eq("IC50").all()
assert df_maob_cleaned["standard_units"].eq("nM").all()
assert df_maob_cleaned["standard_relation"].eq("=").all()

# 5. Identifier validation
assert df_maob_cleaned["molecule_chembl_id"].notna().all()
assert df_maob_cleaned["assay_chembl_id"].notna().all()

# 6. Partition validation
assert (
    len(df_maob_cleaned) + len(df_maob_excluded)
    == 17947
)

# 7. Duplicate candidate validation
duplicate_count = df_maob_cleaned[
    "duplicate_candidate"
].sum()

print("\nB. VALIDATION RESULTS")
print("Missing required columns:", len(missing_columns))
print("Missing activity values:",
      df_maob_cleaned["standard_value"].isna().sum())
print("Non-positive values:",
      (df_maob_cleaned["standard_value"] <= 0).sum())
print("Unique molecules:",
      df_maob_cleaned["molecule_chembl_id"].nunique())
print("Unique assays:",
      df_maob_cleaned["assay_chembl_id"].nunique())
print("Duplicate candidate rows:", duplicate_count)

print("\nC. FINAL STATUS")
print("All validation checks: PASSED")

print("=" * 70)

In [ ]:
import numpy as np

# Preserve original standardized IC50 values
df_maob_cleaned["pIC50"] = (
    9 - np.log10(df_maob_cleaned["standard_value"])
)

# Validate pIC50
assert np.isfinite(df_maob_cleaned["pIC50"]).all()

print("=" * 65)
print("pIC50 TRANSFORMATION — MAO-B")
print("=" * 65)

print("Records:", len(df_maob_cleaned))
print("Missing pIC50:", df_maob_cleaned["pIC50"].isna().sum())

print("\nDescriptive statistics:")
display(df_maob_cleaned["pIC50"].describe())

# Save transformed dataset
df_maob_cleaned.to_csv(
    CLEANED_DATA_DIR / "maob_activity_cleaned.csv",
    index=False
)

print("\nUpdated cleaned dataset saved successfully.")

In [ ]:
#OUTLIER
columns_to_check = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pIC50",
    "document_chembl_id"
]

print("10 LOWEST pIC50")
display(
    df_maob_cleaned
    .sort_values("pIC50")
    [columns_to_check]
    .head(10)
)

print("\n10 HIGHEST pIC50")
display(
    df_maob_cleaned
    .sort_values("pIC50", ascending=False)
    [columns_to_check]
    .head(10)
)

In [ ]:
# Select extreme records
df_extreme = pd.concat([
    df_maob_cleaned.nsmallest(10, "pIC50"),
    df_maob_cleaned.nlargest(10, "pIC50")
]).drop_duplicates()

# Relevant columns for audit
audit_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "document_chembl_id",
    "type",
    "value",
    "units",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pIC50",  # Tambahkan kolom ini
    "pchembl_value",
    "data_validity_comment",
    "activity_comment",
    "bao_endpoint",
    "assay_description",
    "assay_type",
]

# Hanya gunakan kolom yang tersedia
audit_columns = [
    col for col in audit_columns
    if col in df_extreme.columns
]

display(
    df_extreme[audit_columns]
    .sort_values("pIC50")
)

In [ ]:
# Audit all ChEMBL validity comments
validity_audit = (
    df_maob_cleaned[
        df_maob_cleaned["data_validity_comment"].notna()
    ]
    ["data_validity_comment"]
    .value_counts()
    .reset_index()
)

validity_audit.columns = [
    "Validity_Comment",
    "Record_Count"
]

validity_audit["Percentage"] = (
    validity_audit["Record_Count"]
    / len(df_maob_cleaned) * 100
)

print("=" * 65)
print("MAO-B DATA VALIDITY AUDIT")
print("=" * 65)

display(validity_audit)

In [ ]:
flagged_maob = df_maob_cleaned[
    df_maob_cleaned["data_validity_comment"].notna()
].copy()

print("Flagged records:", len(flagged_maob))

display(
    flagged_maob[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "standard_units",
            "pIC50",
            "data_validity_comment",
            "assay_description",
            "document_chembl_id"
        ]
    ].sort_values("pIC50")
)

In [ ]:
# Create a separate quality-controlled copy
df_maob_quality = df_maob_cleaned.copy()

# Flag ChEMBL validity warnings
df_maob_quality["quality_flag"] = (
    df_maob_quality["data_validity_comment"]
    .fillna("No warning")
)

# Create binary indicator
df_maob_quality["is_flagged"] = (
    df_maob_quality["data_validity_comment"].notna()
)

# Summarize quality status
quality_summary = (
    df_maob_quality["quality_flag"]
    .value_counts()
    .reset_index()
)

quality_summary.columns = [
    "Quality_Status",
    "Record_Count"
]

quality_summary["Percentage"] = (
    quality_summary["Record_Count"]
    / len(df_maob_quality) * 100
)

display(quality_summary)

# Save quality-annotated dataset
df_maob_quality.to_csv(
    CLEANED_DATA_DIR / "maob_activity_quality_annotated.csv",
    index=False
)

print("Quality-annotated dataset saved.")

In [ ]:
target_columns = [
    col for col in df_maob_cleaned.columns
    if "target" in col.lower()
    or "assay" in col.lower()
]

print("Available target/assay columns:")
print(target_columns)

In [ ]:
suspect_assay = df_maob_cleaned[
    df_maob_cleaned["assay_chembl_id"].eq("CHEMBL5732981")
]

display(suspect_assay)

In [ ]:
# Audit hubungan assay dengan target
assay_target_audit = (
    df_maob_cleaned
    .groupby(
        [
            "assay_chembl_id",
            "target_chembl_id",
            "target_pref_name",
            "target_organism"
        ],
        dropna=False
    )
    .agg(
        Record_Count=("activity_id", "count"),
        Assay_Description=("assay_description", "first")
    )
    .reset_index()
)

display(
    assay_target_audit[
        assay_target_audit["target_chembl_id"] != "CHEMBL2039"
    ]
)

print("Target distribution:")
display(
    df_maob_cleaned["target_chembl_id"]
    .value_counts(dropna=False)
)

In [ ]:
suspect_assay = df_maob_cleaned[
    df_maob_cleaned["assay_chembl_id"] == "CHEMBL5732981"
].copy()

print("Jumlah record:", len(suspect_assay))

display(
    suspect_assay[
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_description",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "type",
            "standard_type",
            "standard_value",
            "standard_units",
            "pIC50",
            "document_chembl_id"
        ]
    ].head(20)
)

In [ ]:
assay_target_count = (
    df_maob_cleaned
    .groupby("assay_chembl_id")["target_chembl_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Assay dengan lebih dari satu target:")
display(assay_target_count[assay_target_count > 1])

In [ ]:
# Audit distribusi nilai IC50 pada assay mencurigakan
suspect_value_audit = (
    suspect_assay
    .groupby("standard_value")
    .agg(
        Record_Count=("activity_id", "count"),
        Unique_Compounds=("molecule_chembl_id", "nunique"),
        Activity_IDs=("activity_id", list)
    )
    .reset_index()
    .sort_values("Record_Count", ascending=False)
)

display(suspect_value_audit)

print("Jumlah record:", len(suspect_assay))
print(
    "Jumlah senyawa unik:",
    suspect_assay["molecule_chembl_id"].nunique()
)
print(
    "Jumlah nilai IC50 unik:",
    suspect_assay["standard_value"].nunique()
)

In [ ]:
display(
    suspect_assay[
        [
            "assay_chembl_id",
            "assay_description",
            "document_chembl_id",
            "target_chembl_id",
            "target_pref_name",
            "standard_type",
            "standard_value",
            "standard_units",
            "data_validity_comment"
        ]
    ].drop_duplicates()
)

In [ ]:
# Audit nilai asli dan nilai standardisasi
columns_to_audit = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "type",
    "value",
    "units",
    "relation",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pchembl_value",
    "data_validity_comment",
    "activity_comment",
    "document_chembl_id"
]

display(
    suspect_assay[columns_to_audit]
    .sort_values("standard_value")
    .to_string(index=False)
)

In [ ]:
repeated_value = 8776000.0

repeated_records = suspect_assay[
    suspect_assay["standard_value"] == repeated_value
].copy()

print("Jumlah records:", len(repeated_records))
print(
    "Jumlah senyawa unik:",
    repeated_records["molecule_chembl_id"].nunique()
)

display(repeated_records[columns_to_audit])

In [ ]:
# ============================================================
# ASSAY QUALITY REVIEW — MAO-B
# ============================================================

suspect_assay_id = "CHEMBL5732981"

# Identifikasi record berdasarkan assay
mask_suspect = (
    df_maob_cleaned["assay_chembl_id"]
    .eq(suspect_assay_id)
)

# Pisahkan dataset
df_maob_quarantine = df_maob_cleaned[mask_suspect].copy()

df_maob_review = df_maob_cleaned[~mask_suspect].copy()

# Tambahkan status audit
df_maob_quarantine["audit_status"] = (
    "Quarantined - assay description mismatch"
)

df_maob_review["audit_status"] = (
    "Retained - pending general assay review"
)

# Ringkasan
print("ORIGINAL CLEANED:", len(df_maob_cleaned))
print("QUARANTINE:", len(df_maob_quarantine))
print("REVIEW DATASET:", len(df_maob_review))

# Validasi
assert len(df_maob_review) + len(df_maob_quarantine) == len(df_maob_cleaned)

# Simpan
df_maob_quarantine.to_csv(
    CLEANED_DATA_DIR / "maob_assay_quarantine.csv",
    index=False
)

df_maob_review.to_csv(
    CLEANED_DATA_DIR / "maob_activity_review.csv",
    index=False
)

In [ ]:
# Periksa struktur kolom asli
print(df_maob_cleaned.columns.tolist())

# Periksa beberapa kolom yang relevan
display(
    df_maob_cleaned[
        [
            "activity_id",
            "record_id",
            "activity_comment",
            "document_chembl_id",
            "assay_chembl_id"
        ]
    ].head(15)
)

In [ ]:
suspect_assay_id = "CHEMBL5732981"

df_suspect = df_maob_cleaned[
    df_maob_cleaned["assay_chembl_id"] == suspect_assay_id
].copy()

# Periksa tipe data kolom penting
print("DATA TYPES:")
display(
    df_suspect[
        [
            "activity_id",
            "record_id",
            "activity_comment",
            "standard_value",
            "pIC50",
            "data_validity_comment"
        ]
    ].dtypes
)

# Tampilkan seluruh 40 record
audit_columns = [
    "activity_id",
    "record_id",
    "molecule_chembl_id",
    "standard_value",
    "standard_units",
    "pIC50",
    "pchembl_value",
    "data_validity_comment",
    "activity_comment",
    "document_chembl_id",
    "assay_chembl_id",
    "assay_description"
]

display(
    df_suspect[audit_columns]
    .sort_values("standard_value")
    .reset_index(drop=True)
)

In [ ]:
print("JUMLAH RECORD:", len(df_suspect))

print("\nDISTRIBUSI STANDARD VALUE:")
display(
    df_suspect["standard_value"]
    .describe()
    .to_frame()
)

print("\nDISTRIBUSI VALIDITY COMMENT:")
display(
    df_suspect["data_validity_comment"]
    .value_counts(dropna=False)
)

print("\nJUMLAH MOLEKUL UNIK:")
print(df_suspect["molecule_chembl_id"].nunique())

print("\nJUMLAH NILAI STANDARD VALUE UNIK:")
print(df_suspect["standard_value"].nunique())

In [ ]:
assay_id = "CHEMBL5732981"

# Ambil metadata assay dari data mentah
assay_metadata = df_maob_raw[
    df_maob_raw["assay_chembl_id"].eq(assay_id)
]

metadata_columns = [
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "document_chembl_id",
    "document_year",
    "document_journal",
    "src_id",
    "bao_endpoint",
    "bao_label"
]

display(
    assay_metadata[metadata_columns]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
extreme_records = df_suspect[
    df_suspect["standard_value"].eq(8776000)
]

display(
    extreme_records[
        [
            "molecule_chembl_id",
            "activity_id",
            "record_id",
            "type",
            "value",
            "units",
            "standard_type",
            "standard_value",
            "relation",
            "standard_relation",
            "data_validity_comment"
        ]
    ]
)

In [ ]:
# 03.9 - ASSAY-LEVEL QUALITY REVIEW

from pathlib import Path

# Salin seluruh record yang dikarantina
df_maob_quarantine = df_suspect.copy()

df_maob_quarantine["audit_status"] = (
    "Quarantined - assay description and target annotation mismatch"
)

df_maob_quarantine["audit_reason"] = (
    "Assay description mentions LSD1, "
    "while ChEMBL target annotation indicates MAO-B"
)

# Tandai 14 record dengan nilai ekstrem
df_maob_quarantine["extreme_value_flag"] = (
    df_maob_quarantine["standard_value"] == 8776000
)

# Buat dataset provisional
df_maob_review = df_maob_cleaned[
    df_maob_cleaned["assay_chembl_id"] != "CHEMBL5732981"
].copy()

# Simpan hasil
df_maob_quarantine.to_csv(
    CLEANED_DATA_DIR / "maob_assay_quarantine.csv",
    index=False
)

df_maob_review.to_csv(
    CLEANED_DATA_DIR / "maob_activity_provisional.csv",
    index=False
)

print("ASSAY-LEVEL REVIEW SUMMARY")
print("=" * 55)
print("Original cleaned :", len(df_maob_cleaned))
print("Quarantine       :", len(df_maob_quarantine))
print("Extreme records  :", df_maob_quarantine["extreme_value_flag"].sum())
print("Provisional data :", len(df_maob_review))

In [ ]:
# 03.10 - FINAL DATASET VALIDATION

print("=" * 60)
print("FINAL DATASET VALIDATION — MAO-B")
print("=" * 60)

checks = {
    "Original dataset preserved":
        len(df_maob_cleaned) == 6006,

    "Quarantine contains 40 records":
        len(df_maob_quarantine) == 40,

    "Provisional dataset contains 5966 records":
        len(df_maob_review) == 5966,

    "No quarantined assay in provisional dataset":
        not df_maob_review["assay_chembl_id"].eq(
            "CHEMBL5732981"
        ).any(),

    "All endpoints are IC50":
        df_maob_review["standard_type"].eq("IC50").all(),

    "All units are nM":
        df_maob_review["standard_units"].eq("nM").all(),

    "All relations are exact":
        df_maob_review["standard_relation"].eq("=").all(),

    "No missing activity values":
        df_maob_review["standard_value"].notna().all(),

    "All activity values are positive":
        df_maob_review["standard_value"].gt(0).all(),

    "No missing pIC50":
        df_maob_review["pIC50"].notna().all()
}

for name, result in checks.items():
    print(f"[{'PASS' if result else 'FAIL'}] {name}")

print("\nTOTAL CHECKS:", len(checks))
print("PASSED:", sum(checks.values()))
print("FAILED:", len(checks) - sum(checks.values()))

assert all(checks.values()), "Final validation failed!"

print("\nFINAL VALIDATION: PASSED")

In [ ]:
# CHECK ALPHA-SYNUCLEIN ASSAY METADATA

alpha_metadata_path = (
    ALPHA_SYNUCLEIN_DIR /
    "alpha_synuclein_assay_metadata_raw.csv"
)

print("FILE PATH:", alpha_metadata_path)
print("FILE EXISTS:", alpha_metadata_path.is_file())

if alpha_metadata_path.is_file():
    df_alpha_assay_metadata = pd.read_csv(
        alpha_metadata_path
    )

    print("\nDATASET SHAPE:", df_alpha_assay_metadata.shape)
    print("\nCOLUMNS:")
    print(df_alpha_assay_metadata.columns.tolist())

    display(df_alpha_assay_metadata.head())
else:
    print("File belum terdeteksi. Periksa sinkronisasi Google Drive.")

In [ ]:
# ============================================================
# ALPHA-SYNUCLEIN ASSAY METADATA AUDIT
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN ASSAY METADATA AUDIT")
print("=" * 65)

print("\nA. BASIC INFORMATION")
print("Rows:", len(df_alpha_assay_metadata))
print("Columns:", len(df_alpha_assay_metadata.columns))
print("Duplicate rows:", df_alpha_assay_metadata.duplicated().sum())

print("\nB. MISSING VALUES")
missing = (
    df_alpha_assay_metadata.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (
        missing / len(df_alpha_assay_metadata) * 100
    )
})

display(missing_summary[missing_summary["Missing_Count"] > 0])

print("\nC. TARGET DISTRIBUTION")
display(
    df_alpha_assay_metadata["target_chembl_id"]
    .value_counts(dropna=False)
)

print("\nD. ORGANISM DISTRIBUTION")
display(
    df_alpha_assay_metadata["assay_organism"]
    .value_counts(dropna=False)
)

print("\nE. ASSAY TYPE DISTRIBUTION")
display(
    df_alpha_assay_metadata["assay_type"]
    .value_counts(dropna=False)
)

print("\nF. RELATIONSHIP TYPE")
display(
    df_alpha_assay_metadata["relationship_type"]
    .value_counts(dropna=False)
)

print("\nG. BAO LABEL")
display(
    df_alpha_assay_metadata["bao_label"]
    .value_counts(dropna=False)
)

In [ ]:
# Tampilkan seluruh deskripsi assay

assay_descriptions = (
    df_alpha_assay_metadata[
        [
            "assay_chembl_id",
            "description",
            "assay_type",
            "target_chembl_id",
            "relationship_type",
            "confidence_score"
        ]
    ]
    .drop_duplicates()
    .sort_values("assay_chembl_id")
)

display(assay_descriptions)

In [ ]:

# ============================================================
# ALPHA-SYNUCLEIN ACTIVITY AND METADATA INTEGRATION AUDIT
# ============================================================

# Pastikan dataset aktivitas sudah tersedia
if "df_alpha_activity" not in globals():
    df_alpha_activity = pd.read_csv(
        ALPHA_SYNUCLEIN_DIR /
        "alpha_synuclein_activity_raw.csv"
    )

df_alpha_meta = df_alpha_assay_metadata.copy()
df_alpha_raw = df_alpha_activity.copy()

print("=" * 70)
print("ALPHA-SYNUCLEIN ACTIVITY AUDIT")
print("=" * 70)

# A. Basic information
print("\nA. BASIC INFORMATION")
print("Activity records:", len(df_alpha_raw))
print("Activity columns:", len(df_alpha_raw.columns))
print("Assay metadata:", len(df_alpha_meta))
print("Exact duplicate activity rows:",
      df_alpha_raw.duplicated().sum())

# B. Endpoint distribution
print("\nB. STANDARD TYPE DISTRIBUTION")

for col in ["type", "standard_type"]:
    if col in df_alpha_raw.columns:
        print(f"\n{col}:")
        display(
            df_alpha_raw[col]
            .value_counts(dropna=False)
            .to_frame("Record_Count")
        )

# C. Units
print("\nC. ORIGINAL UNITS")

if "units" in df_alpha_raw.columns:
    display(
        df_alpha_raw["units"]
        .value_counts(dropna=False)
        .to_frame("Record_Count")
    )

print("\nSTANDARD UNITS")

if "standard_units" in df_alpha_raw.columns:
    display(
        df_alpha_raw["standard_units"]
        .value_counts(dropna=False)
        .to_frame("Record_Count")
    )

# D. Relation distribution
print("\nD. RELATION DISTRIBUTION")

for col in ["relation", "standard_relation"]:
    if col in df_alpha_raw.columns:
        print(f"\n{col}:")
        display(
            df_alpha_raw[col]
            .value_counts(dropna=False)
            .to_frame("Record_Count")
        )

# E. Assay metadata coverage
print("\nE. ASSAY METADATA COVERAGE")

activity_assays = set(
    df_alpha_raw["assay_chembl_id"].dropna().unique()
)

metadata_assays = set(
    df_alpha_meta["assay_chembl_id"].dropna().unique()
)

matched = activity_assays & metadata_assays
unmatched = activity_assays - metadata_assays

print("Unique activity assays:", len(activity_assays))
print("Unique metadata assays:", len(metadata_assays))
print("Matched assays:", len(matched))
print("Unmatched activity assays:", len(unmatched))

print(
    "Metadata coverage (%):",
    round(len(matched) / len(activity_assays) * 100, 2)
    if activity_assays else 0
)

# F. Target distribution in activity
print("\nF. ACTIVITY TARGET DISTRIBUTION")

if "target_chembl_id" in df_alpha_raw.columns:
    display(
        df_alpha_raw["target_chembl_id"]
        .value_counts(dropna=False)
    )

# G. Join metadata
df_alpha_audit = df_alpha_raw.merge(
    df_alpha_meta[
        [
            "assay_chembl_id",
            "relationship_type",
            "confidence_score",
            "bao_label",
            "assay_type",
            "assay_organism",
            "target_chembl_id"
        ]
    ],
    on="assay_chembl_id",
    how="left",
    suffixes=("", "_metadata"),
    indicator=True
)

print("\nG. METADATA JOIN STATUS")
display(
    df_alpha_audit["_merge"]
    .value_counts()
    .to_frame("Record_Count")
)

print("\nH. RELATIONSHIP TYPE IN ACTIVITY RECORDS")
display(
    df_alpha_audit["relationship_type"]
    .value_counts(dropna=False)
)

print("\nI. BAO LABEL IN ACTIVITY RECORDS")
display(
    df_alpha_audit["bao_label"]
    .value_counts(dropna=False)
)


In [ ]:

# ============================================================
# ALPHA-SYNUCLEIN ASSAY RELATIONSHIP & ENDPOINT AUDIT
# ============================================================

# A. Relationship type per assay
relationship_by_assay = (
    df_alpha_audit.groupby(
        ["assay_chembl_id", "relationship_type"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

print("A. RELATIONSHIP TYPE PER ASSAY")
display(
    relationship_by_assay["relationship_type"]
    .value_counts(dropna=False)
)

print("\nB. ASSAY COUNTS PER RELATIONSHIP")
display(
    relationship_by_assay.groupby("relationship_type")
    ["assay_chembl_id"]
    .nunique()
)

# B. Endpoint by relationship
print("\nC. ENDPOINT BY RELATIONSHIP")

endpoint_relationship = pd.crosstab(
    df_alpha_audit["standard_type"],
    df_alpha_audit["relationship_type"],
    margins=True
)

display(endpoint_relationship)

# C. Endpoint and BAO
print("\nD. ENDPOINT BY BAO LABEL")

endpoint_bao = pd.crosstab(
    df_alpha_audit["standard_type"],
    df_alpha_audit["bao_label"],
    margins=True
)

display(endpoint_bao)

# D. Confidence score
print("\nE. CONFIDENCE SCORE BY RELATIONSHIP")

display(
    df_alpha_audit.groupby(
        "relationship_type",
        dropna=False
    )["confidence_score"]
    .describe()
)

# E. Assay type F inspection
print("\nF. NON-BINDING ASSAY TYPE")

display(
    df_alpha_audit.loc[
        df_alpha_audit["assay_type_metadata"].eq("F"),
        [
            "assay_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "relationship_type",
            "bao_label",
            "description"
        ]
    ].drop_duplicates()
)

# F. Assay descriptions for homologous records
print("\nG. HOMOLOGOUS ASSAY DESCRIPTIONS")

homologous_assays = (
    df_alpha_audit.loc[
        df_alpha_audit["relationship_type"].eq("H"),
        [
            "assay_chembl_id",
            "standard_type",
            "bao_label",
            "confidence_score",
            "description"
        ]
    ]
    .drop_duplicates("assay_chembl_id")
    .sort_values("assay_chembl_id")
)

display(homologous_assays)


In [ ]:

# ============================================================
# FIX: ASSAY TYPE AND DESCRIPTION AUDIT
# ============================================================

print("=" * 70)
print("ASSAY TYPE F — METADATA REVIEW")
print("=" * 70)

# A. Inspect assay type F
df_alpha_assay_F = df_alpha_assay_metadata.loc[
    df_alpha_assay_metadata["assay_type"].eq("F"),
    [
        "assay_chembl_id",
        "description",
        "assay_type",
        "target_chembl_id",
        "relationship_type",
        "confidence_score",
        "bao_label",
        "assay_organism",
        "document_chembl_id"
    ]
].copy()

display(df_alpha_assay_F)

# B. Inspect all homologous assays
print("\n" + "=" * 70)
print("HOMOLOGOUS ASSAY REVIEW")
print("=" * 70)

df_alpha_homologous = (
    df_alpha_assay_metadata.loc[
        df_alpha_assay_metadata["relationship_type"].eq("H"),
        [
            "assay_chembl_id",
            "description",
            "assay_type",
            "target_chembl_id",
            "confidence_score",
            "bao_label",
            "assay_organism",
            "document_chembl_id"
        ]
    ]
    .sort_values("assay_chembl_id")
    .copy()
)

print("Homologous assays:", len(df_alpha_homologous))

display(df_alpha_homologous)

# C. Summarize homologous assay endpoint distribution
print("\n" + "=" * 70)
print("HOMOLOGOUS ASSAY ENDPOINT DISTRIBUTION")
print("=" * 70)

homologous_ids = set(
    df_alpha_homologous["assay_chembl_id"]
)

df_alpha_homologous_activity = df_alpha_raw.loc[
    df_alpha_raw["assay_chembl_id"].isin(homologous_ids)
].copy()

display(
    df_alpha_homologous_activity[
        "standard_type"
    ].value_counts(dropna=False).to_frame("Record_Count")
)


In [ ]:

# ============================================================
# POTENCY ASSAY DESCRIPTION REVIEW
# ============================================================

potency_assay_ids = set(
    df_alpha_raw.loc[
        df_alpha_raw["standard_type"].eq("Potency"),
        "assay_chembl_id"
    ].dropna()
)

df_alpha_potency_review = (
    df_alpha_assay_metadata.loc[
        df_alpha_assay_metadata["assay_chembl_id"].isin(
            potency_assay_ids
        ),
        [
            "assay_chembl_id",
            "description",
            "relationship_type",
            "confidence_score",
            "bao_label",
            "assay_organism",
            "document_chembl_id"
        ]
    ]
    .copy()
)

print("Unique Potency assays:", len(df_alpha_potency_review))

print("\nPOTENCY ASSAY DESCRIPTION SAMPLE")
display(
    df_alpha_potency_review.head(30)
)


In [ ]:
# Ambil metadata assay Direct
df_alpha_direct_metadata = (
    df_alpha_assay_metadata.loc[
        df_alpha_assay_metadata["relationship_type"].eq("D"),
        [
            "assay_chembl_id",
            "description",
            "assay_type",
            "confidence_score",
            "bao_label",
            "assay_organism",
            "document_chembl_id"
        ]
    ]
    .copy()
)

# Hitung jumlah record per assay
direct_counts = (
    df_alpha_raw.loc[
        df_alpha_raw["assay_chembl_id"].isin(
            df_alpha_direct_metadata["assay_chembl_id"]
        )
    ]
    .groupby("assay_chembl_id")
    .agg(
        record_count=("molecule_chembl_id", "size"),
        endpoint_count=("standard_type", "nunique")
    )
    .reset_index()
)

# Gabungkan metadata dan jumlah record
df_alpha_direct_review = (
    df_alpha_direct_metadata
    .merge(
        direct_counts,
        on="assay_chembl_id",
        how="left"
    )
    .sort_values(
        ["record_count", "assay_chembl_id"],
        ascending=[False, True]
    )
)

print("Jumlah Direct assays:",
      len(df_alpha_direct_review))

display(df_alpha_direct_review)

In [ ]:
direct_ids = set(
    df_alpha_direct_metadata["assay_chembl_id"]
)

df_alpha_direct_activity = df_alpha_raw.loc[
    df_alpha_raw["assay_chembl_id"].isin(direct_ids)
].copy()

print("Direct activity records:",
      len(df_alpha_direct_activity))

print("\nEndpoint distribution:")
display(
    df_alpha_direct_activity[
        "standard_type"
    ].value_counts(dropna=False)
)

print("\nEndpoint × BAO label:")
display(
    pd.crosstab(
        df_alpha_direct_activity["standard_type"],
        df_alpha_direct_activity["bao_label"],
        dropna=False
    )
)

In [ ]:
# Distribusi jumlah endpoint dalam setiap assay
display(
    df_alpha_direct_review["endpoint_count"]
    .value_counts()
    .sort_index()
    .rename_axis("Endpoint_Count")
    .reset_index(name="Assay_Count")
)

In [ ]:
target_endpoints = ["IC50", "Ki", "Kd"]

df_alpha_binding_audit = df_alpha_direct_activity.loc[
    df_alpha_direct_activity["standard_type"].isin(
        target_endpoints
    )
].copy()

# Gabungkan metadata assay
assay_columns = [
    "assay_chembl_id",
    "description",
    "bao_label",
    "assay_organism",
    "document_chembl_id"
]

df_alpha_binding_audit = df_alpha_binding_audit.merge(
    df_alpha_assay_metadata[assay_columns],
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

# Ringkasan endpoint dan satuan
display(
    df_alpha_binding_audit.groupby(
        ["standard_type", "bao_label", "standard_units"],
        dropna=False
    ).agg(
        record_count=("molecule_chembl_id", "size"),
        unique_assays=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
display(
    df_alpha_binding_audit.groupby(
        ["standard_type", "standard_relation", "standard_units"],
        dropna=False
    ).agg(
        record_count=("standard_value", "size"),
        missing_value=("standard_value", lambda x: x.isna().sum()),
        unique_assays=("assay_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
for endpoint in ["IC50", "Ki", "Kd"]:
    print(f"\n{'=' * 70}")
    print(f"ASSAY DESCRIPTION — {endpoint}")
    print("=" * 70)

    df_review = (
        df_alpha_binding_audit.loc[
            df_alpha_binding_audit["standard_type"].eq(endpoint),
            [
                "assay_chembl_id",
                "description",
                "bao_label",
                "standard_units",
                "document_chembl_id"
            ]
        ]
        .drop_duplicates("assay_chembl_id")
        .sort_values("assay_chembl_id")
    )

    print("Unique assays:", len(df_review))
    display(df_review)

In [ ]:
# ============================================================
# REBUILD BINDING AUDIT — ALPHA-SYNUCLEIN DIRECT
# ============================================================

target_endpoints = ["IC50", "Ki", "Kd"]

# 1. Ambil aktivitas IC50, Ki, dan Kd
df_alpha_binding_audit = df_alpha_direct_activity.loc[
    df_alpha_direct_activity["standard_type"].isin(target_endpoints)
].copy()

# 2. Siapkan metadata assay
metadata_columns = [
    "assay_chembl_id",
    "description",
    "bao_label",
    "assay_organism",
    "document_chembl_id"
]

df_alpha_metadata_subset = (
    df_alpha_assay_metadata[metadata_columns]
    .drop_duplicates(subset="assay_chembl_id")
    .rename(columns={
        "description": "assay_description_meta",
        "bao_label": "assay_bao_label",
        "assay_organism": "assay_organism_meta",
        "document_chembl_id": "document_chembl_id_meta"
    })
)

# 3. Gabungkan metadata
df_alpha_binding_audit = df_alpha_binding_audit.merge(
    df_alpha_metadata_subset,
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

# 4. Periksa kolom hasil merge
print("SHAPE:", df_alpha_binding_audit.shape)

print("\nMETADATA COVERAGE:")
print(
    df_alpha_binding_audit[
        "assay_bao_label"
    ].notna().value_counts()
)

# 5. Ringkasan endpoint, BAO, dan unit
display(
    df_alpha_binding_audit.groupby(
        ["standard_type", "assay_bao_label", "standard_units"],
        dropna=False
    ).agg(
        record_count=("molecule_chembl_id", "size"),
        unique_assays=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
# ============================================================
# ASSAY DESCRIPTION REVIEW
# ============================================================

for endpoint in ["IC50", "Ki", "Kd"]:

    print(f"\n{'=' * 70}")
    print(f"ASSAY DESCRIPTION — {endpoint}")
    print("=" * 70)

    df_review = (
        df_alpha_binding_audit.loc[
            df_alpha_binding_audit["standard_type"].eq(endpoint),
            [
                "assay_chembl_id",
                "assay_description_meta",
                "assay_bao_label",
                "standard_units",
                "standard_relation",
                "document_chembl_id_meta"
            ]
        ]
        .drop_duplicates("assay_chembl_id")
        .sort_values("assay_chembl_id")
    )

    print("Unique assays:", len(df_review))
    display(df_review)

In [ ]:
# ============================================================
# FULL ASSAY DESCRIPTION REVIEW
# ALPHA-SYNUCLEIN — DIRECT RELATIONSHIP
# ============================================================

import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)

for endpoint in ["IC50", "Ki", "Kd"]:

    print(f"\n{'=' * 100}")
    print(f"FULL ASSAY DESCRIPTION — {endpoint}")
    print("=" * 100)

    df_review = (
        df_alpha_binding_audit.loc[
            df_alpha_binding_audit["standard_type"].eq(endpoint),
            [
                "assay_chembl_id",
                "assay_description_meta",
                "assay_bao_label",
                "standard_units",
                "standard_relation",
                "document_chembl_id_meta"
            ]
        ]
        .drop_duplicates("assay_chembl_id")
        .sort_values("assay_chembl_id")
        .reset_index(drop=True)
    )

    display(df_review)

In [ ]:
# ============================================================
# ASSAY CLASSIFICATION — ALPHA-SYNUCLEIN
# ============================================================

assay_classification = {
    # IC50
    ("IC50", "CHEMBL2065090"): "Binding_affinity",
    ("IC50", "CHEMBL5551998"): "Fibril_depolymerization",
    ("IC50", "CHEMBL5529075"): "Needs_review",

    # Ki
    ("Ki", "CHEMBL2065090"): "Binding_affinity",
    ("Ki", "CHEMBL3594881"): "ThioT_displacement",
    ("Ki", "CHEMBL5256054"): "Fibril_binding",
    ("Ki", "CHEMBL5391420"): "Competitive_binding",
    ("Ki", "CHEMBL6106136"): "Radioligand_tissue_binding",
    ("Ki", "CHEMBL6106205"): "Radioligand_tissue_binding",

    # Kd
    ("Kd", "CHEMBL5241349"): "Needs_review"
}

df_alpha_binding_audit["assay_group"] = [
    assay_classification.get(
        (endpoint, assay_id),
        "Aggregation_inhibition_candidate"
        if endpoint == "IC50"
        else "Binding_affinity_candidate"
    )
    for endpoint, assay_id in zip(
        df_alpha_binding_audit["standard_type"],
        df_alpha_binding_audit["assay_chembl_id"]
    )
]

# Tambahkan status kualitas awal
df_alpha_binding_audit["audit_status"] = "Pending_review"

# ============================================================
# SUMMARY
# ============================================================

display(
    df_alpha_binding_audit.groupby(
        ["standard_type", "assay_group"],
        dropna=False
    ).agg(
        record_count=("molecule_chembl_id", "size"),
        unique_assays=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
# ============================================================
# DATA QUALITY AUDIT — ALPHA-SYNUCLEIN
# ============================================================

df_alpha_binding_quality = df_alpha_binding_audit.copy()

df_alpha_binding_quality["quality_issue"] = ""

# Missing value
mask_missing_value = (
    df_alpha_binding_quality["standard_value"].isna()
)

# Missing unit
mask_missing_unit = (
    df_alpha_binding_quality["standard_units"].isna()
)

# Non-exact relation
mask_censored = (
    df_alpha_binding_quality["standard_relation"].ne("=")
    & df_alpha_binding_quality["standard_relation"].notna()
)

# Non-nM unit
mask_non_nM = (
    df_alpha_binding_quality["standard_units"].notna()
    & df_alpha_binding_quality["standard_units"].ne("nM")
)

# Assign audit labels
df_alpha_binding_quality.loc[
    mask_missing_value, "quality_issue"
] += "Missing_value; "

df_alpha_binding_quality.loc[
    mask_missing_unit, "quality_issue"
] += "Missing_unit; "

df_alpha_binding_quality.loc[
    mask_censored, "quality_issue"
] += "Censored_relation; "

df_alpha_binding_quality.loc[
    mask_non_nM, "quality_issue"
] += "Non_nM_unit; "

# Ringkasan
display(
    df_alpha_binding_quality.groupby(
        ["standard_type", "quality_issue"],
        dropna=False
    ).size().reset_index(name="record_count")
)

# Tampilkan record bermasalah
df_alpha_problem_records = df_alpha_binding_quality.loc[
    df_alpha_binding_quality["quality_issue"].ne("")
].copy()

display(
    df_alpha_problem_records[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "standard_relation",
            "assay_group",
            "quality_issue"
        ]
    ]
)

In [ ]:
df_alpha_binding_quality.to_csv(
    CLEANED_DATA_DIR / "alpha_synuclein_binding_audit.csv",
    index=False
)

df_alpha_problem_records.to_csv(
    CLEANED_DATA_DIR / "alpha_synuclein_binding_problem_records.csv",
    index=False
)

In [ ]:
# ============================================================
# ALPHA-SYNUCLEIN — PROVISIONAL DATASET SELECTION
# ============================================================

import pandas as pd
from pathlib import Path

# Salin agar dataset audit asli tetap utuh
df_alpha_provisional = df_alpha_binding_audit.copy()

# Pastikan nilai aktivitas numerik
df_alpha_provisional["standard_value"] = pd.to_numeric(
    df_alpha_provisional["standard_value"],
    errors="coerce"
)

# Kriteria teknis untuk record yang dapat dipakai
valid_mask = (
    df_alpha_provisional["standard_value"].notna()
    & df_alpha_provisional["standard_value"].gt(0)
    & df_alpha_provisional["standard_units"].eq("nM")
    & df_alpha_provisional["standard_relation"].eq("=")
)

df_alpha_provisional["technical_status"] = "Excluded - technical issue"
df_alpha_provisional.loc[
    valid_mask, "technical_status"
] = "Pass - technical criteria"

# Kandidat dataset IC50 agregasi
df_alpha_ic50_aggregation = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("IC50")
    & df_alpha_provisional["assay_group"].eq(
        "Aggregation_inhibition_candidate"
    )
].copy()

# Kandidat dataset IC50 binding
df_alpha_ic50_binding = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("IC50")
    & df_alpha_provisional["assay_group"].eq("Binding_affinity")
].copy()

# Kandidat dataset IC50 depolymerisasi
df_alpha_ic50_depolymerization = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("IC50")
    & df_alpha_provisional["assay_group"].eq(
        "Fibril_depolymerization"
    )
].copy()

# Dataset Ki: tetap dipisahkan berdasarkan assay_group
df_alpha_ki = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("Ki")
].copy()

# Dataset Kd binding
df_alpha_kd_binding = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("Kd")
    & df_alpha_provisional["assay_group"].eq(
        "Binding_affinity_candidate"
    )
].copy()

# Ringkasan jumlah record
dataset_summary = pd.DataFrame([
    {
        "Dataset": "IC50 Aggregation",
        "Records": len(df_alpha_ic50_aggregation),
        "Assays": df_alpha_ic50_aggregation["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_ic50_aggregation["molecule_chembl_id"].nunique()
    },
    {
        "Dataset": "IC50 Binding",
        "Records": len(df_alpha_ic50_binding),
        "Assays": df_alpha_ic50_binding["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_ic50_binding["molecule_chembl_id"].nunique()
    },
    {
        "Dataset": "IC50 Depolymerization",
        "Records": len(df_alpha_ic50_depolymerization),
        "Assays": df_alpha_ic50_depolymerization["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_ic50_depolymerization["molecule_chembl_id"].nunique()
    },
    {
        "Dataset": "Ki",
        "Records": len(df_alpha_ki),
        "Assays": df_alpha_ki["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_ki["molecule_chembl_id"].nunique()
    },
    {
        "Dataset": "Kd Binding",
        "Records": len(df_alpha_kd_binding),
        "Assays": df_alpha_kd_binding["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_kd_binding["molecule_chembl_id"].nunique()
    }
])

display(dataset_summary)

In [ ]:
alpha_output_dir = CLEANED_DATA_DIR / "Alpha_Synuclein"
alpha_output_dir.mkdir(parents=True, exist_ok=True)

datasets_to_save = {
    "alpha_ic50_aggregation_provisional.csv":
        df_alpha_ic50_aggregation,

    "alpha_ic50_binding_provisional.csv":
        df_alpha_ic50_binding,

    "alpha_ic50_depolymerization_provisional.csv":
        df_alpha_ic50_depolymerization,

    "alpha_ki_provisional.csv":
        df_alpha_ki,

    "alpha_kd_binding_provisional.csv":
        df_alpha_kd_binding
}

for filename, dataframe in datasets_to_save.items():
    dataframe.to_csv(
        alpha_output_dir / filename,
        index=False
    )

dataset_summary.to_csv(
    alpha_output_dir / "alpha_binding_endpoint_summary.csv",
    index=False
)

print("Provisional datasets saved successfully.")

In [ ]:
# ============================================================
# REPEATED MEASUREMENT AUDIT
# ============================================================

datasets = {
    "IC50 Aggregation": df_alpha_ic50_aggregation,
    "IC50 Binding": df_alpha_ic50_binding,
    "IC50 Depolymerization": df_alpha_ic50_depolymerization,
    "Ki": df_alpha_ki,
    "Kd Binding": df_alpha_kd_binding
}

for dataset_name, df in datasets.items():

    print("\n" + "=" * 65)
    print(dataset_name)
    print("=" * 65)

    # Molecules measured more than once
    molecule_counts = (
        df.groupby("molecule_chembl_id")
        .size()
        .sort_values(ascending=False)
    )

    print("Total records:", len(df))
    print("Unique molecules:", df["molecule_chembl_id"].nunique())
    print("Repeated molecules:", (molecule_counts > 1).sum())

    # Assay distribution
    assay_counts = (
        df.groupby("assay_chembl_id")
        .size()
        .sort_values(ascending=False)
    )

    print("\nRecords per assay:")
    display(assay_counts.to_frame("record_count"))

    # Value distribution
    print("\nActivity value summary:")
    display(df["standard_value"].describe())

In [ ]:
# ============================================================
# IC50 BINDING — DATA INTEGRITY CHECK
# ============================================================

print("DATAFRAME SHAPE:")
print(df_alpha_ic50_binding.shape)

print("\nSTANDARD VALUE INFORMATION:")
print(df_alpha_ic50_binding["standard_value"].info())

print("\nMISSING VALUE:")
print(
    df_alpha_ic50_binding["standard_value"].isna().sum()
)

print("\nNON-NUMERIC VALUE:")
print(
    pd.to_numeric(
        df_alpha_ic50_binding["standard_value"],
        errors="coerce"
    ).isna().sum()
)

print("\nSTANDARD VALUE CONTENT:")
display(
    df_alpha_ic50_binding[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "standard_relation"
        ]
    ]
)

In [ ]:
print("TOTAL:", len(df_alpha_ic50_binding))

print(
    "Missing value:",
    df_alpha_ic50_binding["standard_value"].isna().sum()
)

print(
    "Non-positive:",
    (
        df_alpha_ic50_binding["standard_value"] <= 0
    ).sum()
)

print(
    "Non-nM:",
    (
        df_alpha_ic50_binding["standard_units"] != "nM"
    ).sum()
)

print(
    "Non-exact relation:",
    (
        df_alpha_ic50_binding["standard_relation"] != "="
    ).sum()
)

In [ ]:
for name, df in {
    "IC50 Aggregation": df_alpha_ic50_aggregation,
    "Kd Binding": df_alpha_kd_binding
}.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    repeated = (
        df.groupby("molecule_chembl_id")
        .agg(
            record_count=("molecule_chembl_id", "size"),
            assay_count=("assay_chembl_id", "nunique"),
            assay_ids=("assay_chembl_id", "unique")
        )
        .query("record_count > 1")
        .sort_values("record_count", ascending=False)
    )

    display(repeated)

In [ ]:
# ============================================================
# AUDIT INHIBITION DAN ACTIVITY — ALPHA-SYNUCLEIN
# ============================================================

df_alpha_inhibition_activity = df_alpha_binding_audit[
    df_alpha_binding_audit["standard_type"].isin(
        ["Inhibition", "Activity"]
    )
].copy()

print("TOTAL RECORDS:", len(df_alpha_inhibition_activity))

print("\nENDPOINT DISTRIBUTION:")
display(
    df_alpha_inhibition_activity["standard_type"]
    .value_counts()
)

print("\nRELATION DISTRIBUTION:")
display(
    pd.crosstab(
        df_alpha_inhibition_activity["standard_type"],
        df_alpha_inhibition_activity["standard_relation"],
        dropna=False
    )
)

print("\nUNIT DISTRIBUTION:")
display(
    pd.crosstab(
        df_alpha_inhibition_activity["standard_type"],
        df_alpha_inhibition_activity["standard_units"],
        dropna=False
    )
)

print("\nASSAY DISTRIBUTION:")
display(
    df_alpha_inhibition_activity.groupby(
        ["standard_type", "assay_group"],
        dropna=False
    ).agg(
        record_count=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_assays=("assay_chembl_id", "nunique")
    )
)

In [ ]:
# ============================================================
# AUDIT INHIBITION DAN ACTIVITY — ALPHA-SYNUCLEIN
# Menggunakan RAW DATA
# ============================================================

df_alpha_inhibition_activity = df_alpha_raw[
    df_alpha_raw["standard_type"].isin(["Inhibition", "Activity"])
].copy()

print("=" * 65)
print("INHIBITION & ACTIVITY AUDIT — ALPHA-SYNUCLEIN")
print("=" * 65)

print("TOTAL RECORDS:", len(df_alpha_inhibition_activity))

print("\nENDPOINT DISTRIBUTION:")
display(
    df_alpha_inhibition_activity["standard_type"].value_counts()
)

print("\nRELATION DISTRIBUTION:")
display(
    pd.crosstab(
        df_alpha_inhibition_activity["standard_type"],
        df_alpha_inhibition_activity["standard_relation"],
        dropna=False
    )
)

print("\nUNIT DISTRIBUTION:")
display(
    pd.crosstab(
        df_alpha_inhibition_activity["standard_type"],
        df_alpha_inhibition_activity["standard_units"],
        dropna=False
    )
)

print("\nTARGET DISTRIBUTION:")
display(
    df_alpha_inhibition_activity["target_chembl_id"].value_counts(
        dropna=False
    )
)

print("\nASSAY SUMMARY:")
display(
    df_alpha_inhibition_activity.groupby("standard_type").agg(
        record_count=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_assays=("assay_chembl_id", "nunique")
    )
)

In [ ]:
print("RAW DATA SHAPE:", df_alpha_raw.shape)

print("\nSTANDARD_TYPE VALUES:")
display(
    df_alpha_raw["standard_type"]
    .value_counts(dropna=False)
)

print("\nCOLUMN NAMES:")
print(df_alpha_raw.columns.tolist())

In [ ]:
# ============================================================
# ASSAY-LEVEL AUDIT — INHIBITION & ACTIVITY
# ============================================================

# Ambil metadata assay yang relevan
assay_metadata_cols = [
    "assay_chembl_id",
    "description",
    "assay_type",
    "relationship_type",
    "confidence_score",
    "bao_format",
    "bao_label",
    "assay_category"
]

# Gunakan hanya kolom yang tersedia
available_cols = [
    col for col in assay_metadata_cols
    if col in df_alpha_assay_metadata.columns
]

df_inhibition_activity_assay = (
    df_alpha_inhibition_activity
    .merge(
        df_alpha_assay_metadata[available_cols].drop_duplicates(
            subset="assay_chembl_id"
        ),
        on="assay_chembl_id",
        how="left",
        suffixes=("", "_metadata")
    )
)

# Ringkasan per assay dan endpoint
assay_summary = (
    df_inhibition_activity_assay
    .groupby(
        ["standard_type", "assay_chembl_id"],
        dropna=False
    )
    .agg(
        record_count=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        units=("standard_units",
               lambda x: ", ".join(
                   sorted(x.dropna().astype(str).unique())
               )),
        relations=("standard_relation",
                   lambda x: ", ".join(
                       sorted(x.dropna().astype(str).unique())
                   )),
        assay_description=("assay_description", "first"),
        assay_type=("assay_type", "first")
    )
    .reset_index()
)

print("TOTAL ASSAYS:", assay_summary["assay_chembl_id"].nunique())

print("\nASSAY COUNTS PER ENDPOINT:")
print(
    assay_summary.groupby("standard_type")["assay_chembl_id"]
    .nunique()
)

print("\nTOP 20 ASSAYS — INHIBITION:")
display(
    assay_summary[
        assay_summary["standard_type"] == "Inhibition"
    ]
    .sort_values("record_count", ascending=False)
    .head(20)
)

print("\nTOP 20 ASSAYS — ACTIVITY:")
display(
    assay_summary[
        assay_summary["standard_type"] == "Activity"
    ]
    .sort_values("record_count", ascending=False)
    .head(20)
)

In [ ]:
# Assay yang memiliki lebih dari satu standard_type
assay_endpoint_overlap = (
    df_alpha_inhibition_activity
    .groupby("assay_chembl_id")["standard_type"]
    .nunique()
)

overlap_ids = assay_endpoint_overlap[
    assay_endpoint_overlap > 1
].index

print("ASSAY YANG MUNCUL DI LEBIH DARI SATU ENDPOINT:")
print(
    df_alpha_inhibition_activity[
        df_alpha_inhibition_activity["assay_chembl_id"].isin(overlap_ids)
    ][
        ["assay_chembl_id", "standard_type",
         "assay_description"]
    ].drop_duplicates().to_string(index=False)
)

In [ ]:
import re
import pandas as pd

# ============================================================
# PROVISIONAL ASSAY CLASSIFICATION
# ============================================================

def classify_alpha_assay(description):
    if pd.isna(description):
        return "Needs_review"

    desc = str(description).lower()

    # Prioritaskan model biologis khusus
    if "caenorhabditis elegans" in desc or "transgenic" in desc:
        return "In_vivo_model"

    if "neuro2a" in desc or "cell" in desc:
        if "dimer" in desc or "aggregation" in desc:
            return "Cell_based_aggregation"

    if "radioligand" in desc or "displacement of [3h]" in desc:
        return "Radioligand_displacement"

    if "picup" in desc or "oligomer formation" in desc:
        return "Oligomer_formation"

    if "depolymerization" in desc or "disaggregation" in desc:
        return "Fibril_disassembly"

    if (
        "binding affinity" in desc
        or "protein-compound complex" in desc
        or "surface plasmon resonance" in desc
    ):
        return "Binding_affinity"

    if "beta-sheet" in desc or "secondary structure" in desc:
        return "Secondary_structure"

    if (
        "aggregation" in desc
        and any(x in desc for x in ["tht", "thioflavin", "ths", "thioflavin s"])
    ):
        return "Aggregation_inhibition_candidate"

    return "Needs_review"


# Klasifikasikan berdasarkan assay_summary
assay_summary["provisional_assay_group"] = (
    assay_summary["assay_description"]
    .apply(classify_alpha_assay)
)

# ============================================================
# A. JUMLAH ASSAY PER KELOMPOK
# ============================================================

print("ASSAY COUNT PER ENDPOINT AND GROUP")

assay_group_summary = (
    assay_summary
    .groupby(
        ["standard_type", "provisional_assay_group"],
        dropna=False
    )
    .agg(
        assay_count=("assay_chembl_id", "nunique"),
        record_count=("record_count", "sum")
    )
    .reset_index()
)

display(
    assay_group_summary.sort_values(
        ["standard_type", "record_count"],
        ascending=[True, False]
    )
)

# ============================================================
# B. ASSAY YANG PERLU REVIEW MANUAL
# ============================================================

df_assay_review = assay_summary[
    assay_summary["provisional_assay_group"] == "Needs_review"
].copy()

print("\nASSAYS NEEDING MANUAL REVIEW:")
print("Total rows:", len(df_assay_review))

display(
    df_assay_review[
        [
            "standard_type",
            "assay_chembl_id",
            "record_count",
            "units",
            "relations",
            "assay_description"
        ]
    ].sort_values("record_count", ascending=False)
)

# ============================================================
# C. SIMPAN HASIL AUDIT SEMENTARA
# ============================================================

assay_summary.to_csv(
    CLEANED_DATA_DIR / "alpha_inhibition_activity_assay_classification.csv",
    index=False
)

assay_group_summary.to_csv(
    CLEANED_DATA_DIR / "alpha_inhibition_activity_assay_group_summary.csv",
    index=False
)

df_assay_review.to_csv(
    CLEANED_DATA_DIR / "alpha_inhibition_activity_assay_manual_review.csv",
    index=False
)

print("\nProvisional assay classification saved.")

In [ ]:
assay_id = "CHEMBL5330809"

df_check = df_alpha_inhibition_activity[
    df_alpha_inhibition_activity["assay_chembl_id"].eq(assay_id)
].copy()

print("RECORD PER ENDPOINT")
print(df_check["standard_type"].value_counts())

print("\nDETAIL RECORD")
display(
    df_check[
        [
            "molecule_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "value",
            "units",
            "activity_comment"
        ]
    ].sort_values(
        ["molecule_chembl_id", "standard_type"]
    )
)

In [ ]:
assay_id = "CHEMBL5330809"

# Filter assay
mask = df_alpha_inhibition_activity["assay_chembl_id"].eq(assay_id)

# Buat kolom jika belum tersedia
if "quality_issue" not in df_alpha_inhibition_activity.columns:
    df_alpha_inhibition_activity["quality_issue"] = ""

# Tandai record dengan nilai aktivitas kosong
df_alpha_inhibition_activity.loc[mask, "quality_issue"] = (
    "missing_activity_value"
)

# Tampilkan hasil
display(
    df_alpha_inhibition_activity.loc[
        mask,
        [
            "molecule_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "quality_issue"
        ]
    ]
)

In [ ]:
print("KOLOM DATAFRAME:")
for col in df_alpha_inhibition_activity.columns:
    print(col)

In [ ]:
classification_columns = [
    col for col in df_alpha_inhibition_activity.columns
    if any(word in col.lower() for word in
           ["group", "class", "category", "review"])
]

print(classification_columns)

In [ ]:
print("DATAFRAME YANG TERSEDIA:")

for name, obj in list(globals().items()):
    if name.startswith("df_") and hasattr(obj, "columns"):
        if "assay_chembl_id" in obj.columns:
            print(name, obj.shape)

In [ ]:
candidate_names = [
    "df_assay_review",
    "df_alpha_direct_review",
    "df_alpha_potency_review",
    "df_alpha_assay_metadata",
    "df_alpha_direct_metadata",
    "df_review"
]

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        print("\n" + "=" * 70)
        print(f"DATAFRAME: {name}")
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())
        display(obj.head(5))

In [ ]:
print("=" * 70)
print("MAO-B REVIEW COLUMNS")
print("=" * 70)

print(df_maob_review.columns.tolist())

print("\n" + "=" * 70)
print("MAO-B RAW COLUMNS")
print("=" * 70)

print(df_maob_raw.columns.tolist())

In [ ]:
display(
    df_maob_review[
        [
            col for col in [
                "assay_chembl_id",
                "standard_type",
                "standard_units",
                "standard_relation",
                "assay_description",
                "assay_type",
                "quality_issue"
            ]
            if col in df_maob_review.columns
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# MAO-B ASSAY-LEVEL INVENTORY
# ============================================================

assay_columns = [
    "assay_chembl_id",
    "standard_type",
    "standard_units",
    "standard_relation",
    "assay_description",
    "assay_type"
]

df_maob_assay_inventory = (
    df_maob_review
    .groupby(assay_columns, dropna=False)
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

df_maob_assay_inventory = (
    df_maob_assay_inventory
    .sort_values(
        ["standard_type", "record_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("=" * 75)
print("MAO-B ASSAY-LEVEL INVENTORY")
print("=" * 75)

print("Total assay-endpoint combinations:",
      len(df_maob_assay_inventory))

print("Unique assay IDs:",
      df_maob_assay_inventory["assay_chembl_id"].nunique())

display(df_maob_assay_inventory.head(20))

In [ ]:
print("=" * 75)
print("STANDARD TYPE DISTRIBUTION")
print("=" * 75)

display(
    df_maob_assay_inventory
    .groupby("standard_type", dropna=False)
    .agg(
        assay_count=("assay_chembl_id", "nunique"),
        record_count=("record_count", "sum")
    )
    .sort_values("record_count", ascending=False)
)

print("\n" + "=" * 75)
print("STANDARD UNITS DISTRIBUTION")
print("=" * 75)

display(
    df_maob_assay_inventory
    .groupby(
        ["standard_type", "standard_units"],
        dropna=False
    )
    .agg(
        assay_count=("assay_chembl_id", "nunique"),
        record_count=("record_count", "sum")
    )
    .sort_values("record_count", ascending=False)
)

In [ ]:
print("=" * 70)
print("MAO-B STANDARD RELATION AUDIT")
print("=" * 70)

display(
    df_maob_review["standard_relation"]
    .value_counts(dropna=False)
    .rename_axis("standard_relation")
    .reset_index(name="record_count")
)

In [ ]:
import re

df_maob_assay_inventory["description_lower"] = (
    df_maob_assay_inventory["assay_description"]
    .fillna("")
    .str.lower()
)

patterns = {
    "human": r"\bhuman\b",
    "recombinant": r"\brecombinant\b",
    "baboon": r"\bbaboon\b",
    "mouse": r"\bmouse\b|\bmice\b",
    "rat": r"\brat\b",
    "unknown_origin": r"unknown origin",
    "kynuramine": r"kynuramine",
    "p_tyramine": r"p[- ]?tyramine|para[- ]?tyramine",
    "benzylamine": r"benzylamine",
    "mao_a_and_b": r"mao[- ]?a.{0,30}mao[- ]?b|"
                   r"mao[- ]?b.{0,30}mao[- ]?a"
}

for category, pattern in patterns.items():
    mask = df_maob_assay_inventory[
        "description_lower"
    ].str.contains(pattern, regex=True, na=False)

    print(f"{category:20s}: {mask.sum():4d} assays")

In [ ]:
review_patterns = (
    r"unknown origin|baboon|mouse|mice|rat|"
    r"mao[- ]?a"
)

df_maob_assay_attention = (
    df_maob_assay_inventory[
        df_maob_assay_inventory[
            "description_lower"
        ].str.contains(
            review_patterns,
            regex=True,
            na=False
        )
    ]
    .sort_values("record_count", ascending=False)
)

print("Assays requiring additional review:",
      len(df_maob_assay_attention))

display(
    df_maob_assay_attention[
        [
            "assay_chembl_id",
            "standard_type",
            "standard_units",
            "record_count",
            "unique_molecules",
            "assay_description"
        ]
    ].head(30)
)

In [ ]:
import numpy as np

def classify_maob_assay(description):
    desc = str(description).lower()

    has_maoa = bool(re.search(r"\bmao[- ]?a\b", desc))
    has_maob = bool(re.search(r"\bmao[- ]?b\b", desc))

    # 1. Assay yang melibatkan kedua isoform
    if has_maoa and has_maob:
        return "MAO_A_and_B_mixed"

    # 2. Assay MAO-A saja
    if has_maoa and not has_maob:
        return "MAO_A_only"

    # 3. Asal spesies non-human
    if re.search(r"\bbaboon\b|\brat\b|\bmouse\b|\bmice\b", desc):
        return "Non_human"

    # 4. Asal protein tidak diketahui
    if "unknown origin" in desc:
        return "Unknown_origin"

    # 5. Human MAO-B
    if has_maob and "human" in desc:
        return "Human_MAO_B"

    # 6. MAO-B tanpa informasi asal yang cukup
    if has_maob:
        return "MAO_B_origin_unclear"

    return "Needs_manual_review"


df_maob_assay_inventory["assay_category"] = (
    df_maob_assay_inventory["assay_description"]
    .apply(classify_maob_assay)
)

display(
    df_maob_assay_inventory["assay_category"]
    .value_counts()
    .rename_axis("assay_category")
    .reset_index(name="assay_count")
)

In [ ]:
df_maob_assay_summary = (
    df_maob_review
    .merge(
        df_maob_assay_inventory[
            ["assay_chembl_id", "assay_category"]
        ],
        on="assay_chembl_id",
        how="left",
        validate="many_to_one"
    )
    .groupby("assay_category", dropna=False)
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        assay_count=("assay_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(df_maob_assay_summary)

In [ ]:
df_maob_manual_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"].isin([
            "MAO_A_and_B_mixed",
            "MAO_A_only",
            "Non_human",
            "Unknown_origin",
            "MAO_B_origin_unclear",
            "Needs_manual_review"
        ])
    ]
    .sort_values("record_count", ascending=False)
)

display(
    df_maob_manual_review[[
        "assay_chembl_id",
        "assay_category",
        "record_count",
        "unique_molecules",
        "assay_description"
    ]]
)

In [ ]:
# ============================================================
# MAO-B TARGET AUDIT
# ============================================================

# Pilih assay yang masih perlu ditinjau
df_flagged_assays = df_maob_assay_inventory[
    df_maob_assay_inventory["assay_category"].isin([
        "MAO_A_and_B_mixed",
        "Unknown_origin",
        "MAO_B_origin_unclear",
        "Needs_manual_review"
    ])
][["assay_chembl_id", "assay_category"]].copy()

# Hubungkan dengan data aktivitas
df_target_audit = df_maob_review.merge(
    df_flagged_assays,
    on="assay_chembl_id",
    how="inner",
    validate="many_to_one"
)

# Ringkasan target berdasarkan kategori assay
df_target_summary = (
    df_target_audit
    .groupby(
        [
            "assay_category",
            "target_chembl_id",
            "target_pref_name",
            "target_organism"
        ],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assay_category", "record_count"],
        ascending=[True, False]
    )
)

print("=" * 65)
print("MAO-B TARGET AUDIT SUMMARY")
print("=" * 65)

display(df_target_summary)

In [ ]:
# ============================================================
# MAO-B MIXED ASSAY TARGET DETAIL
# ============================================================

df_mixed_target_detail = (
    df_target_audit[
        df_target_audit["assay_category"] == "MAO_A_and_B_mixed"
    ]
    .groupby("assay_chembl_id", dropna=False)
    .agg(
        record_count=("activity_id", "count"),

        target_ids=(
            "target_chembl_id",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        ),

        target_names=(
            "target_pref_name",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        ),

        organisms=(
            "target_organism",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        )
    )
    .reset_index()
)

print("=" * 65)
print("MAO-B MIXED ASSAY TARGET DETAIL")
print("=" * 65)

display(df_mixed_target_detail)

In [ ]:
# ============================================================
# TARGET AUDIT VALIDATION
# ============================================================

print("Total flagged records :", len(df_target_audit))

print(
    "Total mixed records   :",
    (df_target_audit["assay_category"] == "MAO_A_and_B_mixed").sum()
)

print(
    "Unique flagged assays :",
    df_target_audit["assay_chembl_id"].nunique()
)

print(
    "Missing target IDs    :",
    df_target_audit["target_chembl_id"].isna().sum()
)

print(
    "Non-MAO-B target IDs  :",
    (
        df_target_audit["target_chembl_id"].notna()
        & (df_target_audit["target_chembl_id"] != "CHEMBL2039")
    ).sum()
)

In [ ]:
# ============================================================
# MANUAL REVIEW: MIXED MAO-A AND MAO-B ASSAYS
# ============================================================

df_mixed_assay_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"]
        == "MAO_A_and_B_mixed"
    ][
        [
            "assay_chembl_id",
            "assay_type",
            "standard_type",
            "standard_units",
            "record_count",
            "unique_molecules",
            "assay_description"
        ]
    ]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

print("Total mixed assays:", len(df_mixed_assay_review))

display(df_mixed_assay_review)

In [ ]:
# ============================================================
# MANUAL REVIEW: MIXED MAO-B ASSAYS
# ============================================================

retain_assays = [
    "CHEMBL3887028",
    "CHEMBL5729814",
    "CHEMBL5730713",
    "CHEMBL5738570",
    "CHEMBL3707880",
    "CHEMBL5729821",
    "CHEMBL5729878",
    "CHEMBL5733762",
    "CHEMBL5734061",
    "CHEMBL3889174",
    "CHEMBL5730304",
    "CHEMBL3996824",
    "CHEMBL5732528",
    "CHEMBL5736769",
    "CHEMBL3707988",
    "CHEMBL5729884"
]

quarantine_assays = [
    "CHEMBL3707812",
    "CHEMBL5729971",
    "CHEMBL5736053",
    "CHEMBL5731899"
]

exclude_assays = [
    "CHEMBL3708014",
    "CHEMBL5729983"
]

# Assign review decisions
df_maob_review["manual_assay_decision"] = "Not_Mixed_Assay"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(retain_assays),
    "manual_assay_decision"
] = "Provisional_Retain"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(quarantine_assays),
    "manual_assay_decision"
] = "Quarantine"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(exclude_assays),
    "manual_assay_decision"
] = "Exclude_Nonhuman"

# Summary
display(
    df_maob_review["manual_assay_decision"]
    .value_counts()
    .to_frame("record_count")
)

In [ ]:
print("Kolom df_maob_review:")
print(df_maob_review.columns.tolist())

print("\nKolom df_maob_assay_inventory:")
print(df_maob_assay_inventory.columns.tolist())

In [ ]:
# Gabungkan kategori assay ke dataset review
df_decision_audit_data = df_maob_review.merge(
    df_maob_assay_inventory[
        ["assay_chembl_id", "assay_category"]
    ].drop_duplicates(),
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

# Ringkasan keputusan
df_decision_audit = (
    df_decision_audit_data
    .groupby(
        ["assay_category", "manual_assay_decision"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assay_category", "manual_assay_decision"]
    )
)

display(df_decision_audit)

In [ ]:
print("Original records :", len(df_maob_review))
print("Merged records   :", len(df_decision_audit_data))
print(
    "Missing category :",
    df_decision_audit_data["assay_category"].isna().sum()
)

In [ ]:
# ============================================================
# MAO-B ORIGIN UNCLEAR: MANUAL REVIEW
# ============================================================

df_unclear_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"]
        == "MAO_B_origin_unclear"
    ][
        [
            "assay_chembl_id",
            "assay_type",
            "standard_type",
            "standard_units",
            "record_count",
            "unique_molecules",
            "assay_description"
        ]
    ]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

print("Total assays:", len(df_unclear_review))
print("Total records:", df_unclear_review["record_count"].sum())

display(df_unclear_review)

In [ ]:
df_unclear_target = (
    df_decision_audit_data[
        df_decision_audit_data["assay_category"]
        == "MAO_B_origin_unclear"
    ]
    .groupby(
        ["target_chembl_id", "target_pref_name",
         "target_organism", "target_tax_id"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

display(df_unclear_target)

In [ ]:
# MAO-B origin unclear: provisional retention
unclear_assays = [
    "CHEMBL1261744",
    "CHEMBL1059215",
    "CHEMBL5732585",
    "CHEMBL945895",
    "CHEMBL2215741",
    "CHEMBL5730350",
    "CHEMBL5730874",
    "CHEMBL5730999",
    "CHEMBL5731037",
    "CHEMBL5733463",
    "CHEMBL5734386",
    "CHEMBL5736641",
    "CHEMBL706869",
    "CHEMBL1275162",
    "CHEMBL5735933",
    "CHEMBL874045",
    "CHEMBL988334"
]

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(unclear_assays),
    "manual_assay_decision"
] = "Provisional_Retain"

In [ ]:
df_unclear_decision_check = (
    df_maob_review[
        df_maob_review["assay_chembl_id"].isin(unclear_assays)
    ]
    .groupby("manual_assay_decision")
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

display(df_unclear_decision_check)

In [ ]:
df_needs_manual_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"] == "Needs_manual_review"
    ][[
        "assay_chembl_id",
        "assay_type",
        "standard_type",
        "standard_units",
        "record_count",
        "unique_molecules",
        "assay_description"
    ]]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

print("Total assays:", len(df_needs_manual_review))
print("Total records:", df_needs_manual_review["record_count"].sum())

display(df_needs_manual_review)

In [ ]:
# 9 assay: provisional retain
retain_assays = [
    "CHEMBL2394315",
    "CHEMBL1949363",
    "CHEMBL732527",
    "CHEMBL733090",
    "CHEMBL734178",
    "CHEMBL828050",
    "CHEMBL732535",
    "CHEMBL1663829",
    "CHEMBL949319"
]

# 1 assay: quarantine
quarantine_assays = [
    "CHEMBL5303895"
]

# 8 assay: exclude
exclude_assays = [
    "CHEMBL5739141",
    "CHEMBL814332",
    "CHEMBL814335",
    "CHEMBL814336",
    "CHEMBL814333",
    "CHEMBL814334",
    "CHEMBL814337",
    "CHEMBL814339"
]

# Terapkan keputusan
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(retain_assays),
    "manual_assay_decision"
] = "Provisional_Retain"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(quarantine_assays),
    "manual_assay_decision"
] = "Quarantine"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(exclude_assays),
    "manual_assay_decision"
] = "Exclude_Nonhuman"

In [ ]:
df_manual_check = (
    df_maob_review[
        df_maob_review["assay_chembl_id"].isin(
            retain_assays + quarantine_assays + exclude_assays
        )
    ]
    .groupby("manual_assay_decision")
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

display(df_manual_check)

In [ ]:
df_unknown_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"] == "Unknown_origin"
    ][[
        "assay_chembl_id",
        "assay_type",
        "standard_type",
        "standard_units",
        "record_count",
        "unique_molecules",
        "assay_description"
    ]]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

print("Total assays:", len(df_unknown_review))
print("Total records:", df_unknown_review["record_count"].sum())

display(df_unknown_review)

In [ ]:
df_unknown_target = (
    df_decision_audit_data[
        df_decision_audit_data["assay_category"] == "Unknown_origin"
    ]
    .groupby(
        [
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "target_tax_id"
        ],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

display(df_unknown_target)

In [ ]:
# Semua assay kategori Unknown_origin
unknown_assays = df_unknown_review["assay_chembl_id"].tolist()

# Tandai sebagai provisional retain
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(unknown_assays),
    "manual_assay_decision"
] = "Provisional_Retain"

# Tambahkan kolom catatan audit
df_maob_review["assay_review_note"] = ""

# Assay dengan tipe A atau T
unusual_assay_type = [
    "CHEMBL4197202",
    "CHEMBL4321226",
    "CHEMBL4378700",
    "CHEMBL4837696",
    "CHEMBL5393271",
    "CHEMBL5551943"
]

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(unusual_assay_type),
    "assay_review_note"
] = "Review assay type A/T"

# Assay dengan konteks binding atau displacement
unusual_assay_context = [
    "CHEMBL3050772",
    "CHEMBL6089657",
    "CHEMBL5216439"
]

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(unusual_assay_context),
    "assay_review_note"
] = "Review binding/displacement context"

In [ ]:
df_unknown_validation = df_maob_review[
    df_maob_review["assay_chembl_id"].isin(unknown_assays)
]

print("Records:", len(df_unknown_validation))
print(
    "Assays:",
    df_unknown_validation["assay_chembl_id"].nunique()
)
print(
    "\nDecision:"
)
display(
    df_unknown_validation["manual_assay_decision"]
    .value_counts()
)

print("\nAssay review notes:")
display(
    df_unknown_validation[
        df_unknown_validation["assay_review_note"] != ""
    ][[
        "assay_chembl_id",
        "assay_review_note"
    ]].drop_duplicates()
)

In [ ]:
# Gabungkan kategori assay ke dataset review
df_final_audit = df_maob_review.merge(
    df_maob_assay_inventory[
        ["assay_chembl_id", "assay_category"]
    ].drop_duplicates(),
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

# Rekap kategori dan keputusan
df_decision_summary = (
    df_final_audit
    .groupby(
        ["assay_category", "manual_assay_decision"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assay_category", "manual_assay_decision"]
    )
)

display(df_decision_summary)

# Cari keputusan yang masih default
df_pending = df_final_audit[
    df_final_audit["manual_assay_decision"] == "Not_Mixed_Assay"
]

print("Records with default decision:", len(df_pending))
print(
    "Assays with default decision:",
    df_pending["assay_chembl_id"].nunique()
)

display(
    df_pending.groupby("assay_category").agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
# 1. Human MAO-B
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(
        df_maob_assay_inventory.loc[
            df_maob_assay_inventory["assay_category"] == "Human_MAO_B",
            "assay_chembl_id"
        ]
    ),
    "manual_assay_decision"
] = "Retain"


# 2. MAO-A only
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(
        df_maob_assay_inventory.loc[
            df_maob_assay_inventory["assay_category"] == "MAO_A_only",
            "assay_chembl_id"
        ]
    ),
    "manual_assay_decision"
] = "Exclude_Not_Human_MAO_B"


# 3. Non-human
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(
        df_maob_assay_inventory.loc[
            df_maob_assay_inventory["assay_category"] == "Non_human",
            "assay_chembl_id"
        ]
    ),
    "manual_assay_decision"
] = "Exclude_Not_Human_MAO_B"


# 4. Samakan label eksklusi lama agar lebih jelas
df_maob_review["manual_assay_decision"] = (
    df_maob_review["manual_assay_decision"].replace({
        "Exclude_Nonhuman": "Exclude_Not_Human_MAO_B"
    })
)

In [ ]:
print("TOTAL RECORDS:", len(df_maob_review))

display(
    df_maob_review.groupby("manual_assay_decision").agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

print(
    "\nRemaining default decisions:",
    (df_maob_review["manual_assay_decision"] == "Not_Mixed_Assay").sum()
)

In [ ]:
# Pisahkan berdasarkan keputusan akhir
df_maob_retain = df_maob_review[
    df_maob_review["manual_assay_decision"] == "Retain"
].copy()

df_maob_provisional = df_maob_review[
    df_maob_review["manual_assay_decision"] == "Provisional_Retain"
].copy()

df_maob_quarantine = df_maob_review[
    df_maob_review["manual_assay_decision"] == "Quarantine"
].copy()

df_maob_excluded = df_maob_review[
    df_maob_review["manual_assay_decision"] == "Exclude_Not_Human_MAO_B"
].copy()

In [ ]:
datasets = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_provisional,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded
}

for name, df in datasets.items():
    print(f"{name}: {len(df)} records")

In [ ]:
from pathlib import Path

output_dir = Path("results/MAO-B/assay_review")
output_dir.mkdir(parents=True, exist_ok=True)

df_maob_retain.to_csv(
    output_dir / "maob_retain.csv", index=False
)

df_maob_provisional.to_csv(
    output_dir / "maob_provisional.csv", index=False
)

df_maob_quarantine.to_csv(
    output_dir / "maob_quarantine.csv", index=False
)

df_maob_excluded.to_csv(
    output_dir / "maob_excluded.csv", index=False
)

In [ ]:
# Audit struktur dataset utama

print("Shape:", df_maob_retain.shape)

print("\nMissing values:")
display(
    df_maob_retain.isna().sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nDuplicate activity_id:")
print(df_maob_retain["activity_id"].duplicated().sum())

print("\nStandard type:")
print(df_maob_retain["standard_type"].value_counts(dropna=False))

print("\nStandard units:")
print(df_maob_retain["standard_units"].value_counts(dropna=False))

print("\nStandard relation:")
print(df_maob_retain["standard_relation"].value_counts(dropna=False))

In [ ]:
import numpy as np
import pandas as pd

# Pastikan standard_value numerik
df_maob_retain["standard_value"] = pd.to_numeric(
    df_maob_retain["standard_value"],
    errors="coerce"
)

# Statistik deskriptif
print("STATISTIK IC50 (nM)")
display(
    df_maob_retain["standard_value"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# Pemeriksaan nilai tidak valid
print("\nNILAI TIDAK VALID")

print(
    "Missing:",
    df_maob_retain["standard_value"].isna().sum()
)

print(
    "Nilai nol:",
    (df_maob_retain["standard_value"] == 0).sum()
)

print(
    "Nilai negatif:",
    (df_maob_retain["standard_value"] < 0).sum()
)

print(
    "Nilai positif:",
    (df_maob_retain["standard_value"] > 0).sum()
)

# Nilai ekstrem menggunakan IQR
q1 = df_maob_retain["standard_value"].quantile(0.25)
q3 = df_maob_retain["standard_value"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df_maob_extreme_check = df_maob_retain[
    (df_maob_retain["standard_value"] < lower_bound) |
    (df_maob_retain["standard_value"] > upper_bound)
].copy()

print("\nIQR BOUNDS")
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Jumlah kandidat outlier:", len(df_maob_extreme_check))

In [ ]:
# Urutkan kandidat outlier dari IC50 tertinggi
df_maob_extreme_check = df_maob_extreme_check.sort_values(
    "standard_value",
    ascending=False
)

display(
    df_maob_extreme_check[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "standard_units",
            "standard_type",
            "pchembl_value",
            "document_chembl_id"
        ]
    ].head(20)
)

In [ ]:
outlier_by_assay = (
    df_maob_extreme_check
    .groupby("assay_chembl_id")
    .agg(
        outlier_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max")
    )
    .sort_values("outlier_count", ascending=False)
    .reset_index()
)

display(outlier_by_assay.head(20))

In [ ]:
total_by_assay = (
    df_maob_retain
    .groupby("assay_chembl_id")
    .size()
    .rename("total_records")
)

outlier_summary = outlier_by_assay.merge(
    total_by_assay,
    on="assay_chembl_id",
    how="left"
)

outlier_summary["outlier_percentage"] = (
    outlier_summary["outlier_count"]
    / outlier_summary["total_records"]
    * 100
)

display(
    outlier_summary.sort_values(
        "outlier_percentage",
        ascending=False
    ).head(20)
)

In [ ]:
# Ambil metadata assay
assay_metadata = df_maob_assay_inventory[
    [
        "assay_chembl_id",
        "assay_description",
        "assay_type",
        "assay_category"
    ]
].drop_duplicates("assay_chembl_id")

# Gabungkan dengan ringkasan outlier
outlier_assay_detail = outlier_summary.merge(
    assay_metadata,
    on="assay_chembl_id",
    how="left",
    validate="one_to_one"
)

# Tampilkan assay dengan jumlah outlier terbanyak
display(
    outlier_assay_detail.sort_values(
        ["outlier_count", "outlier_percentage"],
        ascending=False
    )[
        [
            "assay_chembl_id",
            "assay_category",
            "assay_type",
            "outlier_count",
            "total_records",
            "outlier_percentage",
            "min_ic50",
            "max_ic50",
            "assay_description"
        ]
    ].head(30)
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Transformasi log10 untuk visualisasi
log_ic50 = np.log10(df_maob_retain["standard_value"])

plt.figure(figsize=(10, 5))
plt.hist(log_ic50, bins=50, edgecolor="black")

plt.xlabel("log10(IC50 in nM)")
plt.ylabel("Frequency")
plt.title("Distribution of MAO-B IC50 Values (Log10 Scale)")

plt.tight_layout()
plt.show()

In [ ]:
# Statistik distribusi IC50 per assay

assay_ic50_summary = (
    df_maob_retain
    .groupby("assay_chembl_id")
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        mean_ic50=("standard_value", "mean"),
        median_ic50=("standard_value", "median"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        std_ic50=("standard_value", "std")
    )
    .reset_index()
)

assay_ic50_summary = assay_ic50_summary.sort_values(
    "median_ic50",
    ascending=False
)

display(assay_ic50_summary.head(20))

In [ ]:
assay_description = (
    df_maob_assay_inventory[
        [
            "assay_chembl_id",
            "assay_description",
            "assay_type",
            "assay_category"
        ]
    ]
    .drop_duplicates("assay_chembl_id")
)

assay_ic50_summary = assay_ic50_summary.merge(
    assay_description,
    on="assay_chembl_id",
    how="left",
    validate="one_to_one"
)

display(
    assay_ic50_summary[
        [
            "assay_chembl_id",
            "record_count",
            "median_ic50",
            "min_ic50",
            "max_ic50",
            "assay_description"
        ]
    ].head(20)
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 6))

plt.hist(
    np.log10(assay_ic50_summary["median_ic50"].dropna()),
    bins=30,
    edgecolor="black"
)

plt.xlabel("log10 Median IC50 (nM)")
plt.ylabel("Number of Assays")
plt.title("Distribution of Median IC50 Across MAO-B Assays")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# Salinan data Retain agar data utama tidak berubah
df_maob_assay_analysis = df_maob_retain.copy()

# Konversi IC50 nM menjadi pIC50
df_maob_assay_analysis["pIC50"] = (
    9 - np.log10(df_maob_assay_analysis["standard_value"])
)

# Ringkasan per assay
assay_distribution = (
    df_maob_assay_analysis
    .groupby("assay_chembl_id")
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        median_ic50=("standard_value", "median"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        median_pIC50=("pIC50", "median"),
        std_pIC50=("pIC50", "std")
    )
    .reset_index()
)

# Tambahkan deskripsi assay
assay_info = (
    df_maob_assay_inventory[
        ["assay_chembl_id", "assay_description"]
    ]
    .drop_duplicates("assay_chembl_id")
)

assay_distribution = assay_distribution.merge(
    assay_info,
    on="assay_chembl_id",
    how="left",
    validate="one_to_one"
)

# Urutkan berdasarkan jumlah data
assay_distribution = assay_distribution.sort_values(
    "record_count", ascending=False
)

display(assay_distribution.head(20))

In [ ]:
import matplotlib.pyplot as plt

# Pilih 15 assay dengan jumlah data terbanyak
selected_assays = (
    assay_distribution[
        assay_distribution["record_count"] >= 5
    ]
    .nlargest(15, "record_count")["assay_chembl_id"]
    .tolist()
)

plot_data = df_maob_assay_analysis[
    df_maob_assay_analysis["assay_chembl_id"].isin(selected_assays)
].copy()

# Siapkan data untuk boxplot
boxplot_data = [
    np.log10(
        plot_data.loc[
            plot_data["assay_chembl_id"] == assay,
            "standard_value"
        ]
    )
    for assay in selected_assays
]

plt.figure(figsize=(14, 7))

plt.boxplot(
    boxplot_data,
    tick_labels=selected_assays,
    showfliers=True
)

plt.xticks(rotation=75, ha="right")
plt.ylabel("log10(IC50 in nM)")
plt.xlabel("Assay ChEMBL ID")
plt.title("IC50 Distribution Across the 15 Most Populated MAO-B Assays")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Hitung jumlah record setiap molekul dalam setiap assay
molecule_assay_counts = (
    df_maob_retain
    .groupby(["molecule_chembl_id", "assay_chembl_id"])
    .agg(
        record_count=("activity_id", "count"),
        unique_ic50=("standard_value", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max")
    )
    .reset_index()
)

# Molekul yang memiliki lebih dari satu record dalam assay yang sama
repeated_within_assay = molecule_assay_counts[
    molecule_assay_counts["record_count"] > 1
].sort_values("record_count", ascending=False)

print("Jumlah kombinasi molekul-assay berulang:",
      len(repeated_within_assay))

display(repeated_within_assay.head(20))

In [ ]:
molecule_across_assays = (
    df_maob_retain
    .groupby("molecule_chembl_id")
    .agg(
        total_records=("activity_id", "count"),
        number_of_assays=("assay_chembl_id", "nunique"),
        number_of_ic50=("standard_value", "nunique")
    )
    .reset_index()
)

repeated_across_assays = (
    molecule_across_assays[
        molecule_across_assays["number_of_assays"] > 1
    ]
    .sort_values("number_of_assays", ascending=False)
)

print("Jumlah molekul yang diuji pada beberapa assay:",
      len(repeated_across_assays))

display(repeated_across_assays.head(20))

In [ ]:
# Periksa nilai maksimum yang sering muncul
top_max_values = (
    df_maob_retain
    .groupby("standard_value")
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(top_max_values.head(20))

# Periksa metadata untuk nilai 61.350 nM
df_max_61350 = df_maob_retain[
    df_maob_retain["standard_value"] == 61350
]

display(
    df_max_61350[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "standard_relation",
            "standard_units",
            "data_validity_comment",
            "activity_comment",
            "document_chembl_id"
        ]
    ]
)

In [ ]:
# Hitung rentang variasi IC50 pada setiap kombinasi molekul-assay
repeated_within_assay = repeated_within_assay.copy()

repeated_within_assay["fold_difference"] = (
    repeated_within_assay["max_ic50"] /
    repeated_within_assay["min_ic50"]
)

repeated_within_assay["absolute_difference"] = (
    repeated_within_assay["max_ic50"] -
    repeated_within_assay["min_ic50"]
)

display(
    repeated_within_assay.sort_values(
        "fold_difference", ascending=False
    ).head(20)
)

In [ ]:
df_chembl8706 = df_maob_retain[
    df_maob_retain["molecule_chembl_id"] == "CHEMBL8706"
].copy()

columns_to_check = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "data_validity_comment",
    "activity_comment",
    "document_chembl_id"
]

display(
    df_chembl8706[columns_to_check]
    .sort_values("assay_chembl_id")
)

In [ ]:
df_61350 = df_maob_retain[
    df_maob_retain["standard_value"] == 61350
].copy()

columns_to_check = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "document_chembl_id",
    "assay_description"
]

# Ambil deskripsi dari inventory jika belum tersedia
if "assay_description" not in df_61350.columns:
    df_61350 = df_61350.merge(
        df_maob_assay_inventory[
            ["assay_chembl_id", "assay_description"]
        ].drop_duplicates("assay_chembl_id"),
        on="assay_chembl_id",
        how="left"
    )

display(df_61350[columns_to_check])

In [ ]:
# Ambil 5 pasangan dengan perbedaan IC50 terbesar
top_pairs = (
    repeated_within_assay
    .sort_values("fold_difference", ascending=False)
    .head(5)[["molecule_chembl_id", "assay_chembl_id"]]
)

# Ambil seluruh record asli dari pasangan tersebut
df_top_pairs_detail = df_maob_retain.merge(
    top_pairs,
    on=["molecule_chembl_id", "assay_chembl_id"],
    how="inner"
)

# Tampilkan kolom penting untuk pemeriksaan
display(
    df_top_pairs_detail[
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "document_chembl_id",
            "document_year",
            "document_journal",
            "assay_description"
        ]
    ].sort_values(
        ["molecule_chembl_id", "assay_chembl_id", "standard_value"]
    )
)

In [ ]:
print(df_maob_retain.columns.tolist())

In [ ]:
# Salin dataset utama agar tidak mengubah dataset asli
df_maob_audit = df_maob_retain.copy()

# Hitung jumlah record pada setiap kombinasi molekul dan assay
df_maob_audit["records_per_molecule_assay"] = (
    df_maob_audit
    .groupby(["molecule_chembl_id", "assay_chembl_id"])["activity_id"]
    .transform("count")
)

# Berikan flag pada kombinasi yang memiliki lebih dari satu record
df_maob_audit["repeated_within_assay"] = (
    df_maob_audit["records_per_molecule_assay"] > 1
)

# Periksa hasil
print("Total record:", len(df_maob_audit))

print(
    "Record dalam kombinasi molekul-assay berulang:",
    df_maob_audit["repeated_within_assay"].sum()
)

display(
    df_maob_audit[
        df_maob_audit["repeated_within_assay"]
    ][
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "document_chembl_id",
            "records_per_molecule_assay"
        ]
    ].head(20)
)

In [ ]:
# Ringkasan kombinasi molekul dan assay
repeated_summary = (
    df_maob_audit
    .groupby(["molecule_chembl_id", "assay_chembl_id"])
    .agg(
        record_count=("activity_id", "count"),
        unique_values=("standard_value", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        document_count=("document_chembl_id", "nunique")
    )
    .reset_index()
)

# Ambil kombinasi yang memiliki lebih dari satu record
repeated_summary = repeated_summary[
    repeated_summary["record_count"] > 1
].copy()

# Kategorikan pola pengulangan
repeated_summary["repeat_category"] = "Different values"

repeated_summary.loc[
    repeated_summary["unique_values"] == 1,
    "repeat_category"
] = "Identical values"

display(
    repeated_summary["repeat_category"]
    .value_counts()
)

display(
    repeated_summary.sort_values(
        "record_count",
        ascending=False
    ).head(20)
)

In [ ]:
# Audit detail record berulang
df_repeat_detail = (
    df_maob_audit[
        df_maob_audit["repeated_within_assay"]
    ]
    .sort_values(
        ["molecule_chembl_id", "assay_chembl_id", "standard_value"]
    )
    [[
        "activity_id",
        "molecule_chembl_id",
        "assay_chembl_id",
        "standard_value",
        "standard_units",
        "standard_relation",
        "document_chembl_id",
        "pchembl_value",
        "data_validity_comment",
        "activity_comment"
    ]]
)

display(df_repeat_detail)

In [ ]:
# Ringkasan sumber pengulangan
repeat_document_summary = (
    df_repeat_detail
    .groupby("assay_chembl_id")
    .agg(
        total_records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_documents=("document_chembl_id", "nunique"),
        unique_ic50_values=("standard_value", "nunique")
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

display(repeat_document_summary.head(20))

In [ ]:
# Identifikasi pasangan molekul-assay yang memiliki lebih dari satu record

df_repeat_detail = (
    df_maob_retain
    .groupby(
        ["assay_chembl_id", "molecule_chembl_id"]
    )
    .agg(
        record_count=("activity_id", "count"),
        unique_ic50=("standard_value", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        document_count=("document_chembl_id", "nunique")
    )
    .reset_index()
)

# Ambil pasangan yang memiliki pengukuran berulang
df_repeat_detail = df_repeat_detail[
    df_repeat_detail["record_count"] > 1
].copy()

# Hitung rentang perbedaan nilai IC50
df_repeat_detail["fold_difference"] = (
    df_repeat_detail["max_ic50"] /
    df_repeat_detail["min_ic50"]
)

# Urutkan berdasarkan perbedaan nilai terbesar
df_repeat_detail = df_repeat_detail.sort_values(
    "fold_difference",
    ascending=False
)

display(df_repeat_detail)

In [ ]:
print("Total pasangan molekul-assay berulang:",
      len(df_repeat_detail))

print("\nKategori pengulangan nilai IC50:")
display(
    df_repeat_detail["unique_ic50"]
    .value_counts()
    .sort_index()
)

In [ ]:
df_repeat_detail["difference_category"] = pd.cut(
    df_repeat_detail["fold_difference"],
    bins=[0, 1.05, 2, 5, float("inf")],
    labels=[
        "Very small difference (<=1.05x)",
        "Moderate difference (1.05-2x)",
        "Large difference (2-5x)",
        "Very large difference (>5x)"
    ],
    include_lowest=True
)

print("DISTRIBUSI PERBEDAAN IC50")
display(
    df_repeat_detail["difference_category"]
    .value_counts()
    .sort_index()
)

print("\nPASANGAN DENGAN PERBEDAAN > 2 KALI")
display(
    df_repeat_detail[
        df_repeat_detail["fold_difference"] > 2
    ].sort_values(
        "fold_difference",
        ascending=False
    )
)

In [ ]:
# Simpan hasil audit pengulangan molekul-assay
df_repeat_detail.to_csv(
    OUTPUT_DIR / "maob_repeated_molecule_assay_audit.csv",
    index=False
)

print("Audit pengulangan berhasil disimpan.")

In [ ]:

# ============================================================
# PROVISIONAL DATA REVIEW — MAO-B
# ============================================================

import pandas as pd
import numpy as np

# 1. Pemeriksaan dasar
print("=" * 65)
print("PROVISIONAL DATA REVIEW — MAO-B")
print("=" * 65)

print("Shape:", df_maob_provisional.shape)
print("Unique molecules:",
      df_maob_provisional["molecule_chembl_id"].nunique())
print("Unique assays:",
      df_maob_provisional["assay_chembl_id"].nunique())

# 2. Distribusi endpoint dan satuan
print("\nA. STANDARD TYPE")
display(df_maob_provisional["standard_type"].value_counts(dropna=False))

print("\nB. STANDARD UNITS")
display(df_maob_provisional["standard_units"].value_counts(dropna=False))

print("\nC. STANDARD RELATION")
display(df_maob_provisional["standard_relation"].value_counts(dropna=False))

# 3. Statistik nilai IC50
print("\nD. STANDARD VALUE SUMMARY")
display(
    df_maob_provisional["standard_value"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)

# 4. Distribusi assay category
assay_category_map = (
    df_maob_assay_inventory[
        ["assay_chembl_id", "assay_category"]
    ]
    .drop_duplicates("assay_chembl_id")
)

provisional_assay_review = df_maob_provisional.merge(
    assay_category_map,
    on="assay_chembl_id",
    how="left"
)

print("\nE. ASSAY CATEGORY DISTRIBUTION")
display(
    provisional_assay_review
    .groupby("assay_category", dropna=False)
    .agg(
        records=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_assays=("assay_chembl_id", "nunique")
    )
    .sort_values("records", ascending=False)
)

# 5. Jumlah record per assay
print("\nF. TOP ASSAYS BY RECORD COUNT")
display(
    provisional_assay_review
    .groupby(
        ["assay_chembl_id", "assay_category"],
        dropna=False
    )
    .agg(
        records=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .sort_values("records", ascending=False)
    .head(20)
)

# 6. Periksa nilai tidak valid
print("\nG. VALUE VALIDITY")
print("Missing standard_value:",
      df_maob_provisional["standard_value"].isna().sum())

print("Non-numeric standard_value:",
      pd.to_numeric(
          df_maob_provisional["standard_value"],
          errors="coerce"
      ).isna().sum())

print("Zero or negative values:",
      (
          pd.to_numeric(
              df_maob_provisional["standard_value"],
              errors="coerce"
          ) <= 0
      ).sum())

# 7. Periksa keputusan manual
if "manual_assay_decision" in df_maob_provisional.columns:
    print("\nH. MANUAL ASSAY DECISION")
    display(
        df_maob_provisional["manual_assay_decision"]
        .value_counts(dropna=False)
    )

if "assay_review_note" in df_maob_provisional.columns:
    print("\nI. ASSAY REVIEW NOTES")
    display(
        df_maob_provisional["assay_review_note"]
        .value_counts(dropna=False)
        .head(20)
    )


In [ ]:

# ============================================================
# MANUAL REVIEW — PROVISIONAL ASSAY EVIDENCE
# ============================================================

# A. Ambil informasi assay yang relevan
review_columns = [
    "assay_chembl_id",
    "assay_category",
    "assay_type",
    "assay_description",
    "record_count",
    "unique_molecules"
]

available_columns = [
    col for col in review_columns
    if col in df_maob_assay_inventory.columns
]

provisional_assays = (
    provisional_assay_review[
        ["assay_chembl_id", "assay_category"]
    ]
    .drop_duplicates()
    .merge(
        df_maob_assay_inventory[
            available_columns
        ].drop_duplicates("assay_chembl_id"),
        on=["assay_chembl_id", "assay_category"],
        how="left"
    )
    .sort_values(["assay_category", "assay_chembl_id"])
)

print("TOTAL PROVISIONAL ASSAYS:",
      provisional_assays["assay_chembl_id"].nunique())

# B. Tampilkan assay yang paling membutuhkan pemeriksaan
priority_categories = [
    "MAO_A_and_B_mixed",
    "MAO_B_origin_unclear",
    "Needs_manual_review"
]

priority_assays = provisional_assays[
    provisional_assays["assay_category"].isin(priority_categories)
]

print("\nPRIORITY ASSAY REVIEW")
display(priority_assays)

# C. Tampilkan record yang memiliki catatan manual
note_mask = (
    df_maob_provisional["assay_review_note"]
    .fillna("")
    .str.strip()
    .ne("")
)

print("\nRECORDS WITH MANUAL REVIEW NOTES:",
      note_mask.sum())

note_columns = [
    col for col in [
        "molecule_chembl_id",
        "assay_chembl_id",
        "assay_review_note",
        "assay_description",
        "assay_type",
        "standard_value",
        "standard_units",
        "standard_relation"
    ]
    if col in df_maob_provisional.columns
]

display(
    df_maob_provisional.loc[note_mask, note_columns]
    .sort_values("assay_chembl_id")
)

# D. Simpan daftar assay untuk pemeriksaan
provisional_assays.to_csv(
    OUTPUT_DIR / "maob_provisional_assay_review.csv",
    index=False
)


In [ ]:

# ============================================================
# TARGET ID VERIFICATION — PROVISIONAL MAO-B
# ============================================================

# Cari kolom identitas target yang tersedia
target_columns = [
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "target_type",
    "assay_chembl_id",
    "molecule_chembl_id"
]

available_target_columns = [
    col for col in target_columns
    if col in df_maob_provisional.columns
]

print("AVAILABLE TARGET COLUMNS:")
print(available_target_columns)

# 1. Distribusi target
if "target_chembl_id" in df_maob_provisional.columns:
    print("\nA. TARGET CHEMBL ID")
    display(
        df_maob_provisional["target_chembl_id"]
        .value_counts(dropna=False)
    )

# 2. Nama target
if "target_pref_name" in df_maob_provisional.columns:
    print("\nB. TARGET NAME")
    display(
        df_maob_provisional["target_pref_name"]
        .value_counts(dropna=False)
    )

# 3. Organisme
if "target_organism" in df_maob_provisional.columns:
    print("\nC. TARGET ORGANISM")
    display(
        df_maob_provisional["target_organism"]
        .value_counts(dropna=False)
    )

# 4. Kombinasi target dan organisme
if {
    "target_chembl_id",
    "target_pref_name",
    "target_organism"
}.issubset(df_maob_provisional.columns):

    print("\nD. TARGET-ORGANISM COMBINATION")

    display(
        df_maob_provisional.groupby(
            [
                "target_chembl_id",
                "target_pref_name",
                "target_organism"
            ],
            dropna=False
        )
        .agg(
            records=("molecule_chembl_id", "size"),
            unique_molecules=("molecule_chembl_id", "nunique"),
            unique_assays=("assay_chembl_id", "nunique")
        )
        .reset_index()
        .sort_values("records", ascending=False)
    )

# 5. Target berdasarkan assay
if "target_chembl_id" in df_maob_provisional.columns:
    print("\nE. ASSAY-TARGET MAPPING")

    display(
        df_maob_provisional.groupby(
            ["assay_chembl_id", "target_chembl_id"],
            dropna=False
        )
        .agg(
            records=("molecule_chembl_id", "size"),
            unique_molecules=("molecule_chembl_id", "nunique")
        )
        .reset_index()
        .sort_values("records", ascending=False)
    )

# 6. Jika kolom target tidak tersedia
if "target_chembl_id" not in df_maob_provisional.columns:
    print(
        "\nTarget ID tidak tersedia dalam dataframe."
        "\nKita perlu mengambil informasi target dari data mentah "
        "atau melakukan pengambilan ulang melalui ChEMBL."
    )


In [ ]:

# ============================================================
# TARGET ID VERIFICATION — PROVISIONAL MAO-B
# ============================================================

print("AVAILABLE TARGET COLUMNS:")
print([
    col for col in [
        "target_chembl_id",
        "target_pref_name",
        "target_organism",
        "target_type"
    ]
    if col in df_maob_provisional.columns
])

target_cols = [
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "target_type"
]

available = [
    col for col in target_cols
    if col in df_maob_provisional.columns
]

if available:
    print("\nA. TARGET DISTRIBUTION")

    for col in available:
        print(f"\n{col}")
        display(
            df_maob_provisional[col]
            .value_counts(dropna=False)
        )

    print("\nB. TARGET COMBINATION")

    if len(available) > 1:
        display(
            df_maob_provisional.groupby(
                available,
                dropna=False
            )
            .size()
            .reset_index(name="record_count")
            .sort_values("record_count", ascending=False)
        )
else:
    print(
        "Target metadata tidak tersedia di dataframe."
        "Perlu mengambilnya dari data mentah atau ChEMBL."
    )


In [ ]:

# ============================================================
# PROVISIONAL ASSAY REVIEW — HUMAN MAO-B
# ============================================================

# Ringkas data pada tingkat assay
df_provisional_assay_review = (
    df_maob_provisional
    .groupby("assay_chembl_id", dropna=False)
    .agg(
        record_count=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        assay_type=("assay_type", "first"),
        assay_description=("assay_description", "first"),
        standard_type=("standard_type", "first"),
        standard_units=("standard_units", "first"),
        standard_relation=("standard_relation", "first"),
        manual_notes=("assay_review_note", 
                      lambda x: "; ".join(
                          sorted(x.dropna().astype(str).unique())
                      ))
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

print("TOTAL ASSAYS:", len(df_provisional_assay_review))
print("TOTAL RECORDS:", df_provisional_assay_review["record_count"].sum())

print("\nASSAY TYPE DISTRIBUTION")
display(
    df_provisional_assay_review["assay_type"]
    .value_counts(dropna=False)
)

print("\nFULL PROVISIONAL ASSAY INVENTORY")
display(df_provisional_assay_review)


In [ ]:
# ============================================================
# CLASSIFY PROVISIONAL MAO-B ASSAYS
# ============================================================

df_maob_assay_classified = df_maob_provisional.copy()

# Assay yang deskripsinya menunjukkan binding/displacement
binding_assay_ids = [
    "CHEMBL3050772",
    "CHEMBL6089657",
    "CHEMBL5216439"
]

# Assay dengan assay_type A/T yang perlu ditinjau
at_assay_ids = [
    "CHEMBL5393271",
    "CHEMBL5551943",
    "CHEMBL4197202",
    "CHEMBL4321226",
    "CHEMBL4378700",
    "CHEMBL4837696"
]

# Klasifikasi berdasarkan assay ID dan deskripsi
def classify_maob_assay(row):
    assay_id = row["assay_chembl_id"]
    description = str(row["assay_description"]).lower()

    if assay_id in binding_assay_ids:
        return "Binding_Displacement"

    if any(keyword in description for keyword in [
        "inhibition",
        "inhibitory",
        "inhibitor",
        "ic50"
    ]):
        return "Functional_Inhibition_Candidate"

    return "Manual_Review"


df_maob_assay_classified["assay_category_final"] = (
    df_maob_assay_classified.apply(
        classify_maob_assay,
        axis=1
    )
)

# Flag khusus untuk assay type A/T
df_maob_assay_classified["assay_type_review_flag"] = (
    df_maob_assay_classified["assay_type"].isin(["A", "T"])
)

# Ringkasan hasil
print("ASSAY CLASSIFICATION")
display(
    df_maob_assay_classified[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nASSAY TYPE REVIEW")
display(
    df_maob_assay_classified.groupby(
        ["assay_category_final", "assay_type_review_flag"]
    ).size().rename("record_count").reset_index()
)

# Periksa semua record yang perlu ditinjau
df_maob_manual_review = (
    df_maob_assay_classified[
        df_maob_assay_classified["assay_category_final"]
        == "Manual_Review"
    ]
)

display(
    df_maob_manual_review[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "standard_type",
            "standard_value",
            "standard_units"
        ]
    ]
)

In [ ]:
# Periksa DataFrame terkait MAO-B yang tersedia
for name in [
    "df_maob_provisional",
    "df_provisional_assay_review",
    "df_manual_assay_review",
    "df_type_flag_check",
    "df_manual_check"
]:
    if name in globals():
        obj = globals()[name]

        print(f"\n{'='*60}")
        print(f"VARIABLE: {name}")
        print("Shape:", obj.shape)
        print("Columns:")
        print(obj.columns.tolist())
    else:
        print(f"\n{name}: BELUM TERDEFINISI")

In [ ]:
if "df_provisional_assay_review" in globals():

    print("SHAPE:", df_provisional_assay_review.shape)

    display(
        df_provisional_assay_review.head()
    )

    print("\nKOLOM:")
    print(df_provisional_assay_review.columns.tolist())

In [ ]:
df_provisional_assay_review.to_csv(
    output_dir / "maob_provisional_assay_inventory.csv",
    index=False
)

print("Provisional assay inventory berhasil disimpan.")

In [ ]:
# Audit hasil provisional assay inventory

print("SHAPE:", df_provisional_assay_review.shape)

print("\nDAFTAR KOLOM:")
print(df_provisional_assay_review.columns.tolist())

print("\nDISTRIBUSI KLASIFIKASI:")
if "assay_category_final" in df_provisional_assay_review.columns:
    print(
        df_provisional_assay_review[
            "assay_category_final"
        ].value_counts(dropna=False)
    )
else:
    print("Kolom assay_category_final belum tersedia.")

In [ ]:
import pandas as pd

print("=" * 70)
print("DATAFRAME YANG MEMILIKI KOLOM KLASIFIKASI")
print("=" * 70)

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = obj.columns.tolist()

        if (
            "assay_category_final" in cols
            or "assay_type_review_flag" in cols
        ):
            print(f"\nVariable: {name}")
            print("Shape:", obj.shape)

            relevant_cols = [
                col for col in [
                    "assay_chembl_id",
                    "assay_category_final",
                    "assay_type_review_flag"
                ]
                if col in cols
            ]

            display(obj[relevant_cols].head())

In [ ]:
df_maob_manual_review.to_csv(
    output_dir / "maob_manual_assay_review.csv",
    index=False
)

print("Manual assay review berhasil disimpan.")
print("Total records:", len(df_maob_manual_review))
print(
    "Unique assays:",
    df_maob_manual_review["assay_chembl_id"].nunique()
)

In [ ]:
print("=" * 60)
print("FINAL CLASSIFICATION AUDIT")
print("=" * 60)

print("Total records:", len(df_maob_assay_classified))

print("\nASSAY CATEGORY:")
print(
    df_maob_assay_classified[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nASSAY TYPE REVIEW FLAG:")
print(
    df_maob_assay_classified[
        "assay_type_review_flag"
    ].value_counts(dropna=False)
)

print("\nUNIQUE ASSAYS:")
print(
    df_maob_assay_classified["assay_chembl_id"].nunique()
)

In [ ]:
# Buat ringkasan klasifikasi pada tingkat assay
df_assay_class_summary = (
    df_maob_assay_classified
    .groupby("assay_chembl_id")
    .agg(
        classification_count=(
            "assay_category_final", "nunique"
        ),
        assay_category_final=(
            "assay_category_final",
            lambda x: x.iloc[0]
            if x.nunique() == 1 else "Conflicting"
        ),
        assay_type_review_flag=(
            "assay_type_review_flag", "max"
        )
    )
    .reset_index()
)

# Integrasikan ke inventory 136 assay
df_provisional_assay_inventory_final = (
    df_provisional_assay_review
    .merge(
        df_assay_class_summary,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
)

print("INVENTORY SHAPE:")
print(df_provisional_assay_inventory_final.shape)

print("\nCLASSIFICATION DISTRIBUTION:")
print(
    df_provisional_assay_inventory_final[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nCONFLICTING CLASSIFICATION:")
print(
    df_provisional_assay_inventory_final[
        df_provisional_assay_inventory_final[
            "assay_category_final"
        ] == "Conflicting"
    ]
)

In [ ]:
df_provisional_assay_inventory_final.to_csv(
    output_dir / "maob_provisional_assay_inventory_classified.csv",
    index=False
)

print("Classified assay inventory berhasil disimpan.")

In [ ]:
# Ambil 4 assay yang masih Manual_Review
df_manual_assay_detail = (
    df_provisional_assay_inventory_final.loc[
        df_provisional_assay_inventory_final[
            "assay_category_final"
        ] == "Manual_Review"
    ].copy()
)

display(
    df_manual_assay_detail[
        [
            "assay_chembl_id",
            "record_count",
            "unique_molecules",
            "assay_type",
            "assay_description",
            "manual_notes"
        ]
    ]
)

In [ ]:
df_flagged_records = df_maob_assay_classified.loc[
    df_maob_assay_classified[
        "assay_type_review_flag"
    ].eq(True)
].copy()

print("FLAGGED RECORDS:", len(df_flagged_records))
print(
    "UNIQUE ASSAYS:",
    df_flagged_records["assay_chembl_id"].nunique()
)

display(
    df_flagged_records[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "standard_value",
            "standard_units",
            "assay_category_final",
            "assay_type_review_flag"
        ]
    ].drop_duplicates()
)

In [ ]:
# Salin dataset agar hasil klasifikasi awal tetap tersedia
df_maob_assay_final = df_maob_assay_classified.copy()

# Simpan klasifikasi awal
df_maob_assay_final["assay_category_original"] = (
    df_maob_assay_final["assay_category_final"]
)

# Tandai keputusan review
df_maob_assay_final["manual_review_decision"] = "Not_Required"

df_maob_assay_final["assay_context_note"] = ""

# Empat assay yang telah ditinjau
manual_assay_decisions = {
    "CHEMBL5732528": (
        "Functional_Inhibition_Candidate",
        "Recombinant human MAO-B enzyme assay"
    ),
    "CHEMBL5735933": (
        "Functional_Inhibition_Candidate",
        "MAO-Glo MAO-B selectivity assay"
    ),
    "CHEMBL3707988": (
        "Functional_Inhibition_Candidate",
        "MAO-A and MAO-B tested under separate reaction conditions"
    ),
    "CHEMBL5729884": (
        "Functional_Inhibition_Candidate",
        "MAO-A and MAO-B tested under separate reaction conditions"
    )
}

for assay_id, (decision, note) in manual_assay_decisions.items():

    mask = (
        df_maob_assay_final["assay_chembl_id"] == assay_id
    )

    df_maob_assay_final.loc[
        mask, "assay_category_final"
    ] = decision

    df_maob_assay_final.loc[
        mask, "manual_review_decision"
    ] = "Reviewed_Retain_As_Functional_Candidate"

    df_maob_assay_final.loc[
        mask, "assay_context_note"
    ] = note

# Tandai sembilan record dengan assay type flag
flag_mask = df_maob_assay_final[
    "assay_type_review_flag"
].eq(True)

df_maob_assay_final.loc[
    flag_mask, "assay_type_review_decision"
] = "Reviewed_Functional_Inhibition_Description"

df_maob_assay_final.loc[
    ~flag_mask, "assay_type_review_decision"
] = "Not_Flagged"

# Audit hasil
print("TOTAL RECORDS:", len(df_maob_assay_final))

print("\nFINAL ASSAY CLASSIFICATION:")
print(
    df_maob_assay_final[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nMANUAL REVIEW DECISIONS:")
print(
    df_maob_assay_final[
        "manual_review_decision"
    ].value_counts()
)

print("\nFLAGGED RECORDS:")
print(
    df_maob_assay_final[
        "assay_type_review_decision"
    ].value_counts()
)

In [ ]:
df_maob_assay_final.to_csv(
    output_dir / "maob_provisional_activity_classified_final.csv",
    index=False
)

print("Final provisional activity classification saved.")

In [ ]:
# Buat ringkasan klasifikasi final per assay
df_assay_final_summary = (
    df_maob_assay_final
    .groupby("assay_chembl_id")
    .agg(
        record_count_final=("assay_chembl_id", "size"),
        unique_molecules_final=("molecule_chembl_id", "nunique"),
        assay_category_final=("assay_category_final", "first"),
        assay_type_reviewed=("assay_type_review_decision", "first")
    )
    .reset_index()
)

# Gabungkan dengan inventory assay sebelumnya
df_maob_assay_inventory_final = (
    df_provisional_assay_review
    .drop(
        columns=["record_count", "unique_molecules"],
        errors="ignore"
    )
    .merge(
        df_assay_final_summary,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
)

print("INVENTORY SHAPE:", df_maob_assay_inventory_final.shape)

print("\nCATEGORY DISTRIBUTION:")
print(
    df_maob_assay_inventory_final[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nMISSING CLASSIFICATION:")
print(
    df_maob_assay_inventory_final[
        "assay_category_final"
    ].isna().sum()
)

print("\nRECORD COUNT CHECK:")
print(
    "Activity records:",
    len(df_maob_assay_final)
)

print(
    "Inventory records:",
    df_maob_assay_inventory_final[
        "record_count_final"
    ].sum()
)

In [ ]:
df_maob_assay_inventory_final.to_csv(
    output_dir / "maob_assay_inventory_final.csv",
    index=False
)

print("Final assay inventory berhasil disimpan.")

In [ ]:
# Rekonsiliasi kategori utama MAO-B

category_counts = {
    "Retain": len(df_maob_retain),
    "Provisional": len(df_maob_assay_final),
    "Quarantine": len(df_maob_quarantine),
    "Excluded": len(df_maob_excluded)
}

print("=" * 60)
print("MAO-B DATA RECONCILIATION")
print("=" * 60)

for category, count in category_counts.items():
    print(f"{category}: {count:,}")

total_selected = sum(category_counts.values())

print("-" * 60)
print("TOTAL:", f"{total_selected:,}")
print("Previous selected count:", "6,006")
print("Difference:", 6006 - total_selected)

In [ ]:
datasets = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded
}

for name, df in datasets.items():
    print(
        f"{name}: "
        f"{len(df):,} records | "
        f"{df['molecule_chembl_id'].nunique():,} unique molecules"
    )

In [ ]:
datasets = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded
}

print("=" * 60)
print("ACTIVITY ID OVERLAP CHECK")
print("=" * 60)

for name, df in datasets.items():
    print(
        f"{name}: {len(df):,} records | "
        f"{df['activity_id'].nunique():,} unique activity IDs | "
        f"{df['activity_id'].duplicated().sum():,} duplicate IDs"
    )

print("\nPAIRWISE OVERLAP")

names = list(datasets.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]

        ids_a = set(datasets[a]["activity_id"].dropna())
        ids_b = set(datasets[b]["activity_id"].dropna())

        print(f"{a} vs {b}: {len(ids_a & ids_b)} overlapping IDs")

In [ ]:
import pandas as pd

print("=" * 70)
print("SEARCHING MAO-B DATAFRAME CANDIDATES")
print("=" * 70)

candidates = []

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame) and "activity_id" in obj.columns:
        candidates.append({
            "variable_name": name,
            "records": len(obj),
            "unique_activity_id": obj["activity_id"].nunique(),
            "duplicate_activity_id": obj["activity_id"].duplicated().sum(),
            "has_molecule_id": "molecule_chembl_id" in obj.columns,
            "has_assay_id": "assay_chembl_id" in obj.columns
        })

df_candidates = pd.DataFrame(candidates)

if not df_candidates.empty:
    df_candidates = df_candidates.sort_values(
        by="records", ascending=False
    ).reset_index(drop=True)

    display(df_candidates)
else:
    print("Tidak ditemukan DataFrame dengan kolom activity_id.")

In [ ]:
# Dataset pembanding utama
df_baseline = df_maob_quality

# Gabungkan ID dari empat kategori final
final_datasets = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded
}

final_ids = set()

for df_part in final_datasets.values():
    final_ids.update(df_part["activity_id"].dropna())

baseline_ids = set(df_baseline["activity_id"].dropna())

# Cari records yang belum masuk kategori final
missing_ids = baseline_ids - final_ids

df_missing_40 = df_baseline[
    df_baseline["activity_id"].isin(missing_ids)
].copy()

# Periksa hubungan dengan suspect records
suspect_ids = set(suspect_assay["activity_id"].dropna())

print("=" * 65)
print("FINAL RECONCILIATION: 6,006 vs 5,966")
print("=" * 65)

print("Baseline records:", len(df_baseline))
print("Baseline unique IDs:", len(baseline_ids))
print("Final categorized IDs:", len(final_ids))
print("Missing IDs:", len(missing_ids))
print("Suspect assay records:", len(suspect_ids))
print("Missing IDs matching suspect:", len(missing_ids & suspect_ids))
print("Suspect IDs outside baseline:", len(suspect_ids - baseline_ids))

print("\nMISSING RECORDS PREVIEW")
display(df_missing_40.head(10))

print("\nMISSING RECORDS COLUMNS")
print(df_missing_40.columns.tolist())

In [ ]:
if len(df_missing_40) > 0:

    print("ASSAY DISTRIBUTION")
    display(
        df_missing_40["assay_chembl_id"]
        .value_counts()
        .head(20)
    )

    print("\nENDPOINT DISTRIBUTION")
    display(
        df_missing_40["standard_type"]
        .value_counts(dropna=False)
    )

    print("\nMOLECULE SUMMARY")
    print(
        "Unique molecules:",
        df_missing_40["molecule_chembl_id"].nunique()
    )

In [ ]:
# Salin data 40 records yang belum terklasifikasi
df_maob_wrong_target = df_missing_40.copy()

# Tambahkan alasan eksklusi
df_maob_wrong_target["exclusion_reason"] = (
    "Wrong_Target_Assay_LSD1"
)

df_maob_wrong_target["review_decision"] = (
    "Confirmed_Exclude"
)

# Gabungkan dengan kategori Excluded
df_maob_excluded_final = pd.concat(
    [
        df_maob_excluded,
        df_maob_wrong_target
    ],
    ignore_index=True
)

print("EXCLUDED DATASET FINAL")
print("Records:", len(df_maob_excluded_final))
print(
    "Unique activity IDs:",
    df_maob_excluded_final["activity_id"].nunique()
)

print("\nEXCLUSION REASON")
display(
    df_maob_excluded_final["exclusion_reason"]
    .value_counts(dropna=False)
)

In [ ]:
print("ORIGINAL EXCLUDED DATASET")
print("Shape:", df_maob_excluded.shape)

print("\nAVAILABLE COLUMNS:")
print(df_maob_excluded.columns.tolist())

print("\nEXISTING EXCLUSION-RELATED COLUMNS:")
exclusion_cols = [
    col for col in df_maob_excluded.columns
    if any(keyword in col.lower() for keyword in [
        "reason", "decision", "review", "exclude", "flag"
    ])
]

display(df_maob_excluded[exclusion_cols].head(10))

print("\nEXCLUSION COLUMN VALUE COUNTS:")
for col in exclusion_cols:
    print(f"\n{col}")
    print(df_maob_excluded[col].value_counts(dropna=False))

In [ ]:
# Salin dataset excluded lama
df_maob_excluded_old = df_maob_excluded.copy()

# Isi alasan eksklusi berdasarkan keputusan manual yang sudah ada
df_maob_excluded_old["exclusion_reason"] = (
    df_maob_excluded_old["manual_assay_decision"]
    .map({
        "Exclude_Not_Human_MAO_B": "Not_Human_MAO_B"
    })
)

# Samakan nama kolom keputusan
df_maob_excluded_old["review_decision"] = (
    df_maob_excluded_old["manual_assay_decision"]
)

# Gabungkan dengan 40 records salah target LSD1
df_maob_excluded_final = pd.concat(
    [df_maob_excluded_old, df_maob_wrong_target],
    ignore_index=True
)

# Pemeriksaan
print("=" * 60)
print("FINAL EXCLUDED DATASET")
print("=" * 60)

print("Total records:", len(df_maob_excluded_final))
print("Unique activity IDs:",
      df_maob_excluded_final["activity_id"].nunique())

print("\nEXCLUSION REASON:")
display(
    df_maob_excluded_final["exclusion_reason"]
    .value_counts(dropna=False)
)

print("\nREVIEW DECISION:")
display(
    df_maob_excluded_final["review_decision"]
    .value_counts(dropna=False)
)

print("\nMISSING EXCLUSION REASON:",
      df_maob_excluded_final["exclusion_reason"].isna().sum())

In [ ]:
from collections import Counter
from itertools import combinations

# =========================================================
# 1. DEFINISI EMPAT KATEGORI FINAL
# =========================================================

maob_categories = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded_final
}

baseline = df_maob_quality

# =========================================================
# 2. RINGKASAN JUMLAH RECORDS
# =========================================================

print("=" * 65)
print("FINAL MAO-B RECONCILIATION")
print("=" * 65)

print(f"Baseline records : {len(baseline)}")
print(f"Baseline unique IDs: {baseline['activity_id'].nunique()}")

print("\nCATEGORY SUMMARY")

for name, df in maob_categories.items():
    print(
        f"{name:15} | Records: {len(df):5} "
        f"| Unique IDs: {df['activity_id'].nunique():5} "
        f"| Missing IDs: {df['activity_id'].isna().sum():3}"
    )

total_records = sum(len(df) for df in maob_categories.values())

print("-" * 65)
print("Total categorized records:", total_records)

# =========================================================
# 3. CEK DUPLIKASI DALAM SETIAP KATEGORI
# =========================================================

print("\nDUPLICATE ACTIVITY IDs WITHIN CATEGORIES")

for name, df in maob_categories.items():
    duplicate_count = df["activity_id"].duplicated().sum()
    print(f"{name:15}: {duplicate_count}")

# =========================================================
# 4. CEK OVERLAP ANTARKATEGORI
# =========================================================

category_ids = {
    name: set(df["activity_id"].dropna())
    for name, df in maob_categories.items()
}

print("\nPAIRWISE OVERLAP")

for (name_a, ids_a), (name_b, ids_b) in combinations(
    category_ids.items(), 2
):
    overlap = ids_a.intersection(ids_b)

    print(f"{name_a} vs {name_b}: {len(overlap)}")

# =========================================================
# 5. CEK COVERAGE TERHADAP BASELINE
# =========================================================

baseline_ids = set(baseline["activity_id"].dropna())

all_category_id_list = [
    activity_id
    for df in maob_categories.values()
    for activity_id in df["activity_id"].dropna()
]

category_id_counts = Counter(all_category_id_list)

categorized_ids = set(category_id_counts.keys())

missing_ids = baseline_ids - categorized_ids
extra_ids = categorized_ids - baseline_ids
cross_category_duplicates = {
    activity_id: count
    for activity_id, count in category_id_counts.items()
    if count > 1
}

print("\nFINAL COVERAGE CHECK")
print("Baseline unique IDs:", len(baseline_ids))
print("Categorized unique IDs:", len(categorized_ids))
print("Missing IDs:", len(missing_ids))
print("Extra IDs:", len(extra_ids))
print("Cross-category duplicate IDs:",
      len(cross_category_duplicates))

# =========================================================
# 6. FINAL STATUS
# =========================================================

final_pass = (
    total_records == len(baseline)
    and all(
        df["activity_id"].notna().all()
        and df["activity_id"].is_unique
        for df in maob_categories.values()
    )
    and not missing_ids
    and not extra_ids
    and not cross_category_duplicates
)

print("\n" + "=" * 65)

if final_pass:
    print("RECONCILIATION: PASSED")
else:
    print("RECONCILIATION: NOT PASSED")

print("=" * 65)

In [ ]:
# =========================================================
# FINAL ASSAY SUBCATEGORY QC
# =========================================================

# Pisahkan berdasarkan klasifikasi assay final
df_maob_functional = df_maob_assay_final[
    df_maob_assay_final["assay_category_final"]
    == "Functional_Inhibition_Candidate"
].copy()

df_maob_binding = df_maob_assay_final[
    df_maob_assay_final["assay_category_final"]
    == "Binding_Displacement"
].copy()

print("=" * 65)
print("PROVISIONAL ASSAY SUBCATEGORY QC")
print("=" * 65)

print("Functional inhibition candidates:", len(df_maob_functional))
print("Binding displacement:", len(df_maob_binding))

# Periksa endpoint, satuan, dan relation
qc_columns = [
    "standard_type",
    "standard_units",
    "standard_relation"
]

for name, df in {
    "FUNCTIONAL INHIBITION": df_maob_functional,
    "BINDING DISPLACEMENT": df_maob_binding
}.items():

    print(f"\n{'=' * 65}")
    print(name)
    print("=" * 65)

    for col in qc_columns:
        print(f"\n{col}:")
        display(
            df[col].value_counts(dropna=False).to_frame("count")
        )

    print("\nUnique endpoint-unit combinations:")
    display(
        df.groupby(
            ["standard_type", "standard_units", "standard_relation"],
            dropna=False
        ).size().reset_index(name="records")
    )

In [ ]:
import numpy as np
import pandas as pd

def validate_ic50(df, dataset_name):

    values = pd.to_numeric(
        df["standard_value"],
        errors="coerce"
    )

    print("=" * 60)
    print(dataset_name)
    print("=" * 60)

    print("Total records:", len(df))
    print("Missing/non-numeric:", values.isna().sum())
    print("Zero values:", (values == 0).sum())
    print("Negative values:", (values < 0).sum())
    print("Infinite values:", np.isinf(values).sum())

    valid = values.notna() & np.isfinite(values) & (values > 0)

    print("Valid positive IC50:", valid.sum())
    print("Invalid IC50:", (~valid).sum())

    if valid.any():
        print("\nIC50 descriptive statistics (nM):")
        display(values[valid].describe().to_frame("IC50_nM"))

    return valid


valid_functional = validate_ic50(
    df_maob_functional,
    "FUNCTIONAL INHIBITION"
)

valid_binding = validate_ic50(
    df_maob_binding,
    "BINDING DISPLACEMENT"
)

In [ ]:
import numpy as np
import pandas as pd

# Validasi nilai IC50 pada dataset utama
values = pd.to_numeric(
    df_maob_retain["standard_value"],
    errors="coerce"
)

valid_retain = (
    values.notna()
    & np.isfinite(values)
    & (values > 0)
)

print("=" * 60)
print("CORE DATASET - MAO-B RETAIN")
print("=" * 60)

print("Total records:", len(df_maob_retain))
print("Missing/non-numeric:", values.isna().sum())
print("Zero values:", (values == 0).sum())
print("Negative values:", (values < 0).sum())
print("Infinite values:", np.isinf(values).sum())
print("Valid positive IC50:", valid_retain.sum())
print("Invalid IC50:", (~valid_retain).sum())

print("\nIC50 descriptive statistics (nM):")
display(
    values[valid_retain]
    .describe()
    .to_frame("IC50_nM")
)

In [ ]:
# Ambil 20 nilai IC50 tertinggi pada functional candidates
cols = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "assay_description",
    "document_chembl_id"
]

available_cols = [
    col for col in cols
    if col in df_maob_functional.columns
]

display(
    df_maob_functional
    .sort_values("standard_value", ascending=False)
    [available_cols]
    .head(20)
)

In [ ]:
# Audit nilai IC50 ekstrem pada core dataset

df_extreme = df_maob_retain[
    df_maob_retain["standard_value"] >= 1_000_000
].copy()

print("TOTAL EXTREME RECORDS:", len(df_extreme))
print(
    "UNIQUE MOLECULES:",
    df_extreme["molecule_chembl_id"].nunique()
)
print(
    "UNIQUE ASSAYS:",
    df_extreme["assay_chembl_id"].nunique()
)

# Ringkasan per assay
assay_extreme_summary = (
    df_extreme
    .groupby("assay_chembl_id")
    .agg(
        records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        mean_ic50=("standard_value", "mean"),
        description=("assay_description", "first"),
        document=("document_chembl_id", "first")
    )
    .sort_values("records", ascending=False)
)

display(assay_extreme_summary)

In [ ]:
# Konsistensi nilai IC50 pada core dataset

print("DATAFRAME SHAPE:", df_maob_retain.shape)

print(
    "MAX IC50:",
    df_maob_retain["standard_value"].max()
)

print(
    "EXTREME RECORDS >= 1,000,000 nM:",
    (df_maob_retain["standard_value"] >= 1_000_000).sum()
)

# Periksa tiga molekul dari tabel sebelumnya
check_molecules = [
    "CHEMBL1766622",
    "CHEMBL2418201",
    "CHEMBL1766620"
]

display(
    df_maob_retain[
        df_maob_retain["molecule_chembl_id"].isin(check_molecules)
    ][[
        "activity_id",
        "molecule_chembl_id",
        "assay_chembl_id",
        "standard_value",
        "standard_units",
        "document_chembl_id"
    ]].sort_values("standard_value", ascending=False)
)

In [ ]:
print("QUARANTINE SHAPE:", df_maob_quarantine.shape)

print("\nMISSING VALUES:")
display(
    df_maob_quarantine.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nREASON DISTRIBUTION:")
for col in [
    "quarantine_reason",
    "manual_review_decision",
    "review_decision",
    "assay_category_final"
]:
    if col in df_maob_quarantine.columns:
        print(f"\n{col}:")
        display(df_maob_quarantine[col].value_counts(dropna=False))

print("\nSAMPLE QUARANTINE RECORDS:")
display(
    df_maob_quarantine.head(10)
)

In [ ]:
# 1. Periksa kolom yang tersedia
print("KOLOM TERKAIT REVIEW:")
print([
    col for col in df_maob_quarantine.columns
    if any(x in col.lower() for x in [
        "reason", "review", "decision", "category",
        "flag", "target"
    ])
])

# 2. Jumlah records dan assay unik
print("\nTOTAL RECORDS:", len(df_maob_quarantine))
print(
    "UNIQUE ACTIVITY ID:",
    df_maob_quarantine["activity_id"].nunique()
)
print(
    "UNIQUE ASSAY:",
    df_maob_quarantine["assay_chembl_id"].nunique()
)

# 3. Distribusi keputusan yang sudah tersedia
for col in ["manual_assay_decision", "assay_review_note"]:
    if col in df_maob_quarantine.columns:
        print(f"\n{col.upper()}")
        display(
            df_maob_quarantine[col]
            .value_counts(dropna=False)
        )

# 4. Ringkasan berdasarkan assay
assay_summary = (
    df_maob_quarantine
    .groupby("assay_chembl_id", dropna=False)
    .agg(
        total_records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        endpoint=("standard_type", "unique"),
        units=("standard_units", "unique"),
        relation=("standard_relation", "unique"),
        target=("target_pref_name", "unique"),
        organism=("target_organism", "unique"),
        assay_description=("assay_description", "first"),
        document=("document_chembl_id", "unique")
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

display(assay_summary)

In [ ]:
# Kolom penting untuk audit per record
cols = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "assay_type",
    "type",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "bao_endpoint",
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "document_chembl_id",
    "activity_comment",
    "standard_flag",
    "manual_assay_decision",
    "assay_review_note"
]

cols = [c for c in cols if c in df_maob_quarantine.columns]

# A. Distribusi nilai per assay
display(
    df_maob_quarantine
    .groupby("assay_chembl_id")
    .agg(
        records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        min_IC50=("standard_value", "min"),
        median_IC50=("standard_value", "median"),
        max_IC50=("standard_value", "max"),
        missing_activity_comment=("activity_comment", lambda x: x.isna().sum())
    )
)

# B. Periksa apakah dua assay 52-record menggunakan molekul yang sama
assay_a = set(
    df_maob_quarantine.loc[
        df_maob_quarantine["assay_chembl_id"] == "CHEMBL3707812",
        "molecule_chembl_id"
    ]
)

assay_b = set(
    df_maob_quarantine.loc[
        df_maob_quarantine["assay_chembl_id"] == "CHEMBL5729971",
        "molecule_chembl_id"
    ]
)

print("Molekul pada assay pertama:", len(assay_a))
print("Molekul pada assay kedua:", len(assay_b))
print("Molekul yang sama:", len(assay_a & assay_b))
print("Hanya assay pertama:", len(assay_a - assay_b))
print("Hanya assay kedua:", len(assay_b - assay_a))

# C. Tampilkan seluruh record dari tiga assay yang perlu ditinjau
display(
    df_maob_quarantine[
        df_maob_quarantine["assay_chembl_id"].isin([
            "CHEMBL5736053",
            "CHEMBL5303895",
            "CHEMBL5731899"
        ])
    ][cols]
    .sort_values(["assay_chembl_id", "standard_value"])
)

In [ ]:
# Ambil dua assay yang dicurigai memiliki data sama
assay_1 = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == "CHEMBL3707812"
].copy()

assay_2 = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == "CHEMBL5729971"
].copy()

# Bandingkan nilai aktivitas per molekul
comparison = assay_1[
    ["molecule_chembl_id", "standard_value"]
].merge(
    assay_2[
        ["molecule_chembl_id", "standard_value"]
    ],
    on="molecule_chembl_id",
    how="outer",
    suffixes=("_assay_1", "_assay_2"),
    indicator=True
)

# Periksa kesamaan nilai
comparison["same_value"] = (
    comparison["standard_value_assay_1"]
    == comparison["standard_value_assay_2"]
)

print("TOTAL MOLECULES:", len(comparison))
print("\nMERGE STATUS:")
display(comparison["_merge"].value_counts())

print("\nVALUE COMPARISON:")
display(comparison["same_value"].value_counts(dropna=False))

print("\nMOLECULES WITH DIFFERENT VALUES:")
display(
    comparison[
        comparison["same_value"] == False
    ]
)

In [ ]:
print("ASSAY 1 ACTIVITY IDs:")
print(assay_1["activity_id"].nunique())

print("\nASSAY 2 ACTIVITY IDs:")
print(assay_2["activity_id"].nunique())

print("\nIDENTICAL MOLECULE-VALUE PAIRS:")
print(
    comparison[
        comparison["same_value"] == True
    ].shape[0]
)

In [ ]:
# Periksa jumlah activity ID
print("ASSAY 1 ACTIVITY IDs:", assay_1["activity_id"].nunique())
print("ASSAY 2 ACTIVITY IDs:", assay_2["activity_id"].nunique())

# Bandingkan atribut tambahan
comparison_detail = assay_1[
    [
        "molecule_chembl_id",
        "activity_id",
        "standard_value",
        "standard_type",
        "standard_relation",
        "standard_units",
        "pchembl_value",
        "document_chembl_id"
    ]
].merge(
    assay_2[
        [
            "molecule_chembl_id",
            "activity_id",
            "standard_value",
            "standard_type",
            "standard_relation",
            "standard_units",
            "pchembl_value",
            "document_chembl_id"
        ]
    ],
    on="molecule_chembl_id",
    suffixes=("_assay_1", "_assay_2")
)

columns_to_compare = [
    "standard_value",
    "standard_type",
    "standard_relation",
    "standard_units",
    "pchembl_value",
    "document_chembl_id"
]

for col in columns_to_compare:
    comparison_detail[f"{col}_identical"] = (
        comparison_detail[f"{col}_assay_1"]
        == comparison_detail[f"{col}_assay_2"]
    )

print("\nATTRIBUTE COMPARISON:")
for col in columns_to_compare:
    flag = f"{col}_identical"
    print(f"{col}: {comparison_detail[flag].sum()}/52 identical")

display(comparison_detail.head())

In [ ]:
duplicate_assays = [
    "CHEMBL3707812",
    "CHEMBL5729971"
]

mask_duplicate = df_maob_quarantine[
    "assay_chembl_id"
].isin(duplicate_assays)

df_maob_quarantine.loc[
    mask_duplicate,
    "assay_review_note"
] = (
    "Suspected duplicate assay records: "
    "52 identical molecule-value pairs across two assay IDs; "
    "same endpoint, relation, units, pChEMBL values, "
    "and document. Retained in quarantine pending source verification."
)

print("ANNOTATED RECORDS:", mask_duplicate.sum())

display(
    df_maob_quarantine.loc[
        mask_duplicate,
        [
            "assay_chembl_id",
            "molecule_chembl_id",
            "activity_id",
            "assay_review_note"
        ]
    ].head()
)

In [ ]:
assay_id = "CHEMBL5736053"

df_audit = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == assay_id
].copy()

print("TOTAL RECORDS:", len(df_audit))
print("UNIQUE MOLECULES:", df_audit["molecule_chembl_id"].nunique())
print("UNIQUE ACTIVITY IDs:", df_audit["activity_id"].nunique())

# Informasi assay
display(
    df_audit[
        [
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "document_chembl_id"
        ]
    ].drop_duplicates()
)

# Seluruh aktivitas
display(
    df_audit[
        [
            "activity_id",
            "molecule_chembl_id",
            "type",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "activity_comment",
            "manual_assay_decision",
            "assay_review_note"
        ]
    ].sort_values("standard_value")
)

In [ ]:
assay_id = "CHEMBL5736053"

mask = (
    df_maob_quarantine["assay_chembl_id"] == assay_id
)

df_maob_quarantine.loc[
    mask, "assay_review_note"
] = (
    "Target specificity unclear: assay description covers "
    "both human MAO-A and MAO-B using MAO-Glo. "
    "The available records do not establish whether these "
    "12 IC50 values specifically correspond to MAO-B. "
    "Retained in quarantine pending source-level verification."
)

print(
    "ANNOTATED RECORDS:",
    df_maob_quarantine.loc[mask].shape[0]
)

display(
    df_maob_quarantine.loc[
        mask,
        [
            "assay_chembl_id",
            "activity_id",
            "molecule_chembl_id",
            "standard_value",
            "assay_review_note"
        ]
    ]
)

In [ ]:
assay_id = "CHEMBL5303895"

df_audit = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == assay_id
].copy()

print("TOTAL RECORDS:", len(df_audit))

display(
    df_audit[
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "document_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "activity_comment",
            "manual_assay_decision",
            "assay_review_note"
        ]
    ]
)

In [ ]:
assay_id = "CHEMBL5303895"

mask = (
    df_maob_quarantine["assay_chembl_id"] == assay_id
)

df_maob_quarantine.loc[
    mask, "assay_review_note"
] = (
    "Insufficient assay information: description only states "
    "'Cross screening panel'. The available metadata does not "
    "establish the experimental method or confirm MAO-B-specific "
    "functional inhibition. Retained in quarantine pending "
    "source-level verification."
)

print("ANNOTATED RECORDS:", mask.sum())

In [ ]:
assay_id = "CHEMBL5731899"

df_audit = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == assay_id
].copy()

print("TOTAL RECORDS:", len(df_audit))

display(
    df_audit[
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "document_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "activity_comment",
            "manual_assay_decision",
            "assay_review_note"
        ]
    ]
)

In [ ]:
assay_id = "CHEMBL5731899"

mask = (
    df_maob_quarantine["assay_chembl_id"] == assay_id
)

df_maob_quarantine.loc[
    mask, "manual_assay_decision"
] = "Reviewed_Retain_As_Functional_Candidate"

df_maob_quarantine.loc[
    mask, "assay_review_note"
] = (
    "Reviewed: assay description documents functional "
    "inhibition testing using recombinant human MAO-A "
    "and MAO-B, with separate enzyme addition to wells "
    "and concentration-dependent fluorescence measurement. "
    "ChEMBL target annotation maps this record to human MAO-B "
    "(CHEMBL2039). Retain as a functional inhibition candidate; "
    "individual result-to-enzyme mapping remains subject "
    "to source-level verification."
)

print("UPDATED RECORDS:", mask.sum())

display(
    df_maob_quarantine.loc[
        mask,
        [
            "activity_id",
            "assay_chembl_id",
            "manual_assay_decision",
            "assay_review_note"
        ]
    ]
)

In [ ]:
# ============================================================
# MOVE REVIEWED RECORD FROM QUARANTINE TO PROVISIONAL
# ============================================================

activity_id_to_move = 27086882

# Ambil record yang sudah disetujui
df_move = df_maob_quarantine[
    df_maob_quarantine["activity_id"] == activity_id_to_move
].copy()

assert len(df_move) == 1, (
    f"Expected 1 record, found {len(df_move)}"
)

# Pastikan keputusan manual benar
assert (
    df_move["manual_assay_decision"].iloc[0]
    == "Reviewed_Retain_As_Functional_Candidate"
)

# Tambahkan klasifikasi provisional
df_move["assay_category_final"] = (
    "Functional_Inhibition_Candidate"
)

df_move["manual_review_decision"] = (
    "Reviewed_Retain_As_Functional_Candidate"
)

df_move["assay_type_review_decision"] = (
    "Reviewed_Functional_Inhibition_Description"
)

# Samakan struktur kolom sebelum penggabungan
for col in df_maob_assay_final.columns:
    if col not in df_move.columns:
        df_move[col] = pd.NA

for col in df_move.columns:
    if col not in df_maob_assay_final.columns:
        df_maob_assay_final[col] = pd.NA

df_move = df_move[df_maob_assay_final.columns]

# Tambahkan ke provisional
df_maob_assay_final = pd.concat(
    [df_maob_assay_final, df_move],
    ignore_index=True
)

# Keluarkan dari quarantine
df_maob_quarantine = df_maob_quarantine[
    df_maob_quarantine["activity_id"]
    != activity_id_to_move
].copy()

print("MOVED RECORDS:", len(df_move))
print("PROVISIONAL RECORDS:", len(df_maob_assay_final))
print("QUARANTINE RECORDS:", len(df_maob_quarantine))

In [ ]:
print("PROVISIONAL CATEGORY DISTRIBUTION")
display(
    df_maob_assay_final[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nQUARANTINE ASSAY DISTRIBUTION")
display(
    df_maob_quarantine[
        "assay_chembl_id"
    ].value_counts()
)

print("\nCHECK ACTIVITY ID")
print(
    "Moved ID in provisional:",
    (
        df_maob_assay_final["activity_id"]
        == activity_id_to_move
    ).sum()
)

print(
    "Moved ID in quarantine:",
    (
        df_maob_quarantine["activity_id"]
        == activity_id_to_move
    ).sum()
)

In [ ]:
# ============================================================
# FINAL RECONCILIATION - MAO-B
# ============================================================

# 1. Gabungkan seluruh kategori
category_frames = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded_final
}

# 2. Baseline activity IDs
baseline_ids = set(
    df_maob_quality["activity_id"].dropna()
)

# 3. Audit setiap kategori
all_category_ids = []
category_summary = []

for category, df in category_frames.items():

    ids = df["activity_id"].dropna()

    category_summary.append({
        "category": category,
        "records": len(df),
        "unique_activity_ids": ids.nunique(),
        "duplicate_activity_ids": ids.duplicated().sum(),
        "missing_activity_ids": df["activity_id"].isna().sum()
    })

    all_category_ids.extend(ids.tolist())

# 4. Gabungkan seluruh ID
from collections import Counter

category_id_counts = Counter(all_category_ids)

categorized_ids = set(category_id_counts.keys())

# 5. Periksa missing dan extra IDs
missing_ids = baseline_ids - categorized_ids
extra_ids = categorized_ids - baseline_ids

# 6. Periksa overlap antarkategori
overlap_results = {}

category_names = list(category_frames.keys())

for i in range(len(category_names)):
    for j in range(i + 1, len(category_names)):

        name_a = category_names[i]
        name_b = category_names[j]

        ids_a = set(
            category_frames[name_a]["activity_id"].dropna()
        )

        ids_b = set(
            category_frames[name_b]["activity_id"].dropna()
        )

        overlap_results[
            f"{name_a} vs {name_b}"
        ] = len(ids_a.intersection(ids_b))

# 7. Tampilkan hasil
print("=" * 65)
print("FINAL CATEGORY SUMMARY")
print("=" * 65)

display(pd.DataFrame(category_summary))

print("\nBASELINE RECORDS:", len(df_maob_quality))
print("BASELINE UNIQUE IDs:", len(baseline_ids))

print("\nCATEGORIZED RECORDS:", len(all_category_ids))
print("CATEGORIZED UNIQUE IDs:", len(categorized_ids))

print("\nMISSING IDs:", len(missing_ids))
print("EXTRA IDs:", len(extra_ids))

print("\nDUPLICATE IDs ACROSS CATEGORIES:",
      sum(count > 1 for count in category_id_counts.values()))

print("\nPAIRWISE OVERLAP:")
for pair, count in overlap_results.items():
    print(f"{pair}: {count}")

# 8. Final validation
no_internal_duplicates = all(
    item["duplicate_activity_ids"] == 0
    for item in category_summary
)

no_missing_ids = all(
    item["missing_activity_ids"] == 0
    for item in category_summary
)

no_overlap = all(
    value == 0 for value in overlap_results.values()
)

reconciliation_passed = (
    len(baseline_ids) == len(categorized_ids)
    and len(missing_ids) == 0
    and len(extra_ids) == 0
    and no_internal_duplicates
    and no_missing_ids
    and no_overlap
)

print("\n" + "=" * 65)

if reconciliation_passed:
    print("FINAL RECONCILIATION: PASSED")
else:
    print("FINAL RECONCILIATION: FAILED")

print("=" * 65)

In [ ]:
# ============================================================
# FINAL QC — PROVISIONAL MAO-B
# ============================================================

df_maob_functional = df_maob_assay_final[
    df_maob_assay_final["assay_category_final"]
    == "Functional_Inhibition_Candidate"
].copy()

df_maob_binding = df_maob_assay_final[
    df_maob_assay_final["assay_category_final"]
    == "Binding_Displacement"
].copy()

print("=" * 65)
print("PROVISIONAL DATASET QC")
print("=" * 65)

print("Total provisional:", len(df_maob_assay_final))
print("Functional candidates:", len(df_maob_functional))
print("Binding displacement:", len(df_maob_binding))

print("\nSTANDARD TYPE:")
print(df_maob_assay_final["standard_type"].value_counts(dropna=False))

print("\nSTANDARD UNITS:")
print(df_maob_assay_final["standard_units"].value_counts(dropna=False))

print("\nSTANDARD RELATION:")
print(df_maob_assay_final["standard_relation"].value_counts(dropna=False))

print("\nNUMERIC VALIDATION:")

values = pd.to_numeric(
    df_maob_assay_final["standard_value"],
    errors="coerce"
)

print("Missing/non-numeric:", values.isna().sum())
print("Zero:", (values == 0).sum())
print("Negative:", (values < 0).sum())
print("Infinite:", np.isinf(values).sum())
print("Valid positive:", (values.gt(0) & np.isfinite(values)).sum())

print("\nCATEGORY DISTRIBUTION:")
print(df_maob_assay_final["assay_category_final"].value_counts())

print("\nFUNCTIONAL UNIQUE ACTIVITY IDs:",
      df_maob_functional["activity_id"].nunique())

print("BINDING UNIQUE ACTIVITY IDs:",
      df_maob_binding["activity_id"].nunique())

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# 1. FINAL ANNOTATION CHECK
# ============================================================

print("=" * 65)
print("FINAL ANNOTATION CHECK")
print("=" * 65)

# Quarantine
quarantine_notes = (
    df_maob_quarantine["assay_review_note"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Quarantine records:", len(df_maob_quarantine))
print("Quarantine without review note:", (quarantine_notes == "").sum())

# Excluded
excluded_reasons = (
    df_maob_excluded_final["exclusion_reason"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("\nExcluded records:", len(df_maob_excluded_final))
print("Excluded without reason:", (excluded_reasons == "").sum())

# Functional vs Binding
functional_ids = set(df_maob_functional["activity_id"])
binding_ids = set(df_maob_binding["activity_id"])

print("\nFunctional-Binding overlap:", len(functional_ids & binding_ids))

# Assertions
assert (quarantine_notes != "").all(), "Ada Quarantine tanpa review note."
assert (excluded_reasons != "").all(), "Ada Excluded tanpa exclusion reason."
assert len(functional_ids & binding_ids) == 0, "Ada overlap Functional-Binding."

print("\nANNOTATION CHECK: PASSED")


# ============================================================
# 2. SAVE FINAL PREPROCESSING OUTPUTS
# ============================================================

output_dir = Path("OUTPUTS/01_Cleaned_Data")
output_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "maob_retain_final.csv": df_maob_retain,
    "maob_provisional_final.csv": df_maob_assay_final,
    "maob_functional_inhibition_candidates.csv": df_maob_functional,
    "maob_binding_displacement.csv": df_maob_binding,
    "maob_quarantine_final.csv": df_maob_quarantine,
    "maob_excluded_final.csv": df_maob_excluded_final,
}

for filename, dataframe in datasets.items():
    dataframe.to_csv(output_dir / filename, index=False)

print("\n" + "=" * 65)
print("SAVED DATASETS")
print("=" * 65)

for filename in datasets:
    print(filename)


# ============================================================
# 3. READ-BACK VERIFICATION
# ============================================================

print("\n" + "=" * 65)
print("READ-BACK VERIFICATION")
print("=" * 65)

for filename, original_df in datasets.items():
    saved_df = pd.read_csv(output_dir / filename)

    passed = (
        len(saved_df) == len(original_df)
        and saved_df["activity_id"].nunique()
            == original_df["activity_id"].nunique()
    )

    print(
        f"{filename}: "
        f"{len(saved_df)} records | "
        f"{'PASSED' if passed else 'FAILED'}"
    )

    assert passed, f"Read-back verification failed: {filename}"

print("\nALL FILES SAVED AND VERIFIED")